<a href="https://colab.research.google.com/github/wamo12/FinRL/blob/master/Copy_of_Last_copy_of_FRTBMKTRISKipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install ydata-profiling

In [ ]:
!pip install stockdex -U --no-cache-dir

In [ ]:
import pandas as pd
import numpy as np
import os
from stockdex import Ticker
import matplotlib.pyplot as plt
import seaborn as sns
from dateutil import parser
from datetime import datetime, timedelta, date
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.6f' % x)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

In [ ]:
tickers_list = ["^GSPC", "^N225", "^FTSE", "^HSI", "^STOXX50E", "000001.SS", "^KS11", "XAGG.TO", "GD=F", "BZ=F", "GC=F", "GBPUSD=X", "USDJPY=X", "EURUSD=X", "USDCHF=X"]
all_data = {}

for ticker_symbol in tickers_list:
    print(f"Fetching data for {ticker_symbol}...")
    try:
        ticker = Ticker(ticker=ticker_symbol)
        price_data = ticker.yahoo_api_price(range='5y', dataGranularity='1d')

        if not isinstance(price_data, pd.DataFrame):
            data = pd.DataFrame(price_data)
        else:
            data = price_data

        start_date_filter = '2022-01-01'
        end_date_filter = '2026-06-30'

        data['timestamp'] = pd.to_datetime(data['timestamp'])

        filtered_data = data[(data['timestamp'] >= pd.to_datetime(start_date_filter)) &
                             (data['timestamp'] <= pd.to_datetime(end_date_filter))]

        # Removed .tail(500) to keep all data within the specified date range
        final_data = filtered_data.sort_values('timestamp', ascending=True)

        all_data[ticker_symbol] = final_data
        print(f"Finished fetching data for {ticker_symbol}.")
    except Exception as e:
        print(f"Failed to fetch data for {ticker_symbol}: {e}")

print("\nData for all specified tickers has been fetched and stored in the 'all_data' dictionary.")

Fetching data for ^GSPC...
Finished fetching data for ^GSPC.
Fetching data for ^N225...
Finished fetching data for ^N225.
Fetching data for ^FTSE...
Finished fetching data for ^FTSE.
Fetching data for ^HSI...
Finished fetching data for ^HSI.
Fetching data for ^STOXX50E...
Finished fetching data for ^STOXX50E.
Fetching data for 000001.SS...
Finished fetching data for 000001.SS.
Fetching data for ^KS11...
Finished fetching data for ^KS11.
Fetching data for XAGG.TO...
Finished fetching data for XAGG.TO.
Fetching data for GD=F...
Finished fetching data for GD=F.
Fetching data for BZ=F...
Finished fetching data for BZ=F.
Fetching data for GC=F...
Finished fetching data for GC=F.
Fetching data for GBPUSD=X...
Finished fetching data for GBPUSD=X.
Fetching data for USDJPY=X...
Finished fetching data for USDJPY=X.
Fetching data for EURUSD=X...
Finished fetching data for EURUSD=X.
Fetching data for USDCHF=X...
Finished fetching data for USDCHF=X.

Data for all specified tickers has been fetched 

In [ ]:
combined_df_list = []

for ticker_symbol, df_data in all_data.items():
    # Ensure 'timestamp' is in datetime format and normalize to date only
    if 'timestamp' in df_data.columns:
        df_data['timestamp'] = pd.to_datetime(df_data['timestamp']).dt.normalize()

    # Add a 'ticker' column to identify the source of the data
    df_data['ticker'] = ticker_symbol
    combined_df_list.append(df_data)

# Concatenate all DataFrames into a single DataFrame
# Use 'pd.concat' and 'ignore_index=True' to reset the index and avoid duplicates
combined_df = pd.concat(combined_df_list, ignore_index=True)

# Display the first few rows of the combined DataFrame
print("Combined DataFrame head:")
display(combined_df.head())

print(f"\nTotal rows in combined DataFrame: {len(combined_df)}")
print(f"Unique tickers in combined DataFrame: {combined_df['ticker'].nunique()}")

Combined DataFrame head:


,timestamp,volume,close,open,high,low,currency,timezone,exchangeTimezoneName,exchangeName,instrumentType,ticker
0,2022-01-03,3831020000.000000,4796.560059,4778.140137,4796.640137,4758.169922,USD,EDT,America/New_York,SNP,INDEX,^GSPC
1,2022-01-04,4683170000.000000,4793.540039,4804.509766,4818.620117,4774.270020,USD,EDT,America/New_York,SNP,INDEX,^GSPC
2,2022-01-05,4887960000.000000,4700.580078,4787.990234,4797.700195,4699.439941,USD,EDT,America/New_York,SNP,INDEX,^GSPC
3,2022-01-06,4295280000.000000,4696.049805,4693.390137,4725.009766,4671.259766,USD,EDT,America/New_York,SNP,INDEX,^GSPC
4,2022-01-07,4181510000.000000,4677.029785,4697.660156,4707.950195,4662.740234,USD,EDT,America/New_York,SNP,INDEX,^GSPC



Total rows in combined DataFrame: 16961
Unique tickers in combined DataFrame: 15


In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16961 entries, 0 to 16960
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   timestamp             16961 non-null  datetime64[ns]
 1   volume                16931 non-null  float64       
 2   close                 16931 non-null  float64       
 3   open                  16931 non-null  float64       
 4   high                  16931 non-null  float64       
 5   low                   16931 non-null  float64       
 6   currency              16961 non-null  object        
 7   timezone              16961 non-null  object        
 8   exchangeTimezoneName  16961 non-null  object        
 9   exchangeName          16961 non-null  object        
 10  instrumentType        16961 non-null  object        
 11  ticker                16961 non-null  object        
dtypes: datetime64[ns](1), float64(5), object(6)
memory usage: 1.6+ MB


In [ ]:
# Ensure 'timestamp' is set as the index for the combined_df before pivoting
# Also, ensure 'timestamp' is a datetime object and normalize to date only
combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp']).dt.normalize()
combined_df_pivot = combined_df.set_index('timestamp')

# Pivot the DataFrame
# The index will be the 'timestamp', columns will be the 'ticker', and values will be the 'close' price
# Use 'mean' as aggfunc to handle cases where multiple entries might exist for a single day after normalization
close_prices_df = combined_df_pivot.pivot_table(index=combined_df_pivot.index, columns='ticker', values='close', aggfunc='mean')

# Display the first few rows of the new DataFrame
print("Close prices for all tickers:")
display(close_prices_df.head())

# Display information about the new DataFrame to confirm its structure
print("\nInfo about the pivoted DataFrame:")
close_prices_df.info()

Close prices for all tickers:


ticker,000001.SS,BZ=F,EURUSD=X,GBPUSD=X,GC=F,GD=F,USDCHF=X,USDJPY=X,XAGG.TO,^FTSE,^GSPC,^HSI,^KS11,^N225,^STOXX50E
timestamp,,,,,,,,,,,,,,,
2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078



Info about the pivoted DataFrame:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1305 entries, 2022-01-03 to 2026-06-30
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   000001.SS  1084 non-null   float64
 1   BZ=F       1128 non-null   float64
 2   EURUSD=X   1167 non-null   float64
 3   GBPUSD=X   1167 non-null   float64
 4   GC=F       1127 non-null   float64
 5   GD=F       1126 non-null   float64
 6   USDCHF=X   1167 non-null   float64
 7   USDJPY=X   1167 non-null   float64
 8   XAGG.TO    1126 non-null   float64
 9   ^FTSE      1131 non-null   float64
 10  ^GSPC      1125 non-null   float64
 11  ^HSI       1099 non-null   float64
 12  ^KS11      1095 non-null   float64
 13  ^N225      1097 non-null   float64
 14  ^STOXX50E  1125 non-null   float64
dtypes: float64(15)
memory usage: 163.1 KB


In [ ]:
close_prices_df.to_csv('close_prices.csv', index=True)
print("close_prices_df saved to close_prices.csv")

close_prices_df saved to close_prices.csv


In [ ]:
#df=pd.read_csv(r"/content/close_prices (1).csv" , index_col=0 , parse_dates=True, dayfirst=True)

In [ ]:
df=pd.read_csv("close_prices.csv")

In [ ]:
df.head()

,timestamp,000001.SS,BZ=F,EURUSD=X,GBPUSD=X,GC=F,GD=F,USDCHF=X,USDJPY=X,XAGG.TO,^FTSE,^GSPC,^HSI,^KS11,^N225,^STOXX50E
0,2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
1,2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2,2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
3,2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
4,2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078


In [ ]:
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.normalize()
else:
    print("No 'timestamp' column found. Assuming the index contains date information.")
    # If 'timestamp' is the index, convert and normalize it
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df.index = pd.to_datetime(df.index).dt.normalize()

# Display the info and head to confirm the change
print("DataFrame info after timestamp normalization:")
df.info()
print("\nDataFrame head after timestamp normalization:")
display(df.head())

DataFrame info after timestamp normalization:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1305 entries, 0 to 1304
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  1305 non-null   datetime64[ns]
 1   000001.SS  1084 non-null   float64       
 2   BZ=F       1128 non-null   float64       
 3   EURUSD=X   1167 non-null   float64       
 4   GBPUSD=X   1167 non-null   float64       
 5   GC=F       1127 non-null   float64       
 6   GD=F       1126 non-null   float64       
 7   USDCHF=X   1167 non-null   float64       
 8   USDJPY=X   1167 non-null   float64       
 9   XAGG.TO    1126 non-null   float64       
 10  ^FTSE      1131 non-null   float64       
 11  ^GSPC      1125 non-null   float64       
 12  ^HSI       1099 non-null   float64       
 13  ^KS11      1095 non-null   float64       
 14  ^N225      1097 non-null   float64       
 15  ^STOXX50E  1125 non-null   float64       
d

,timestamp,000001.SS,BZ=F,EURUSD=X,GBPUSD=X,GC=F,GD=F,USDCHF=X,USDJPY=X,XAGG.TO,^FTSE,^GSPC,^HSI,^KS11,^N225,^STOXX50E
0,2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
1,2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2,2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
3,2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
4,2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078


In [ ]:
if 'timestamp' in df.columns:
    df = df.rename(columns={'timestamp': 'date'})

##-- convert column names to lowercase and replace spaces with underscores (excluding the index)
current_columns = df.columns.tolist()
df.columns = [col.replace(' ', '_').lower() for col in current_columns]

# Process the 'date' component
if 'date' in df.columns:
    # If 'date' is still a column, convert it to datetime and set as index
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
elif df.index.name == 'date': # if 'date' is already the index
    # If date is already the datetime index, ensure it's datetime (if not already)
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df.index = pd.to_datetime(df.index)

# Ensure the index name is explicitly 'date'
df.index.name = 'date'

df.head()

,000001.ss,bz=f,eurusd=x,gbpusd=x,gc=f,gd=f,usdchf=x,usdjpy=x,xagg.to,^ftse,^gspc,^hsi,^ks11,^n225,^stoxx50e
date,,,,,,,,,,,,,,,
2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1305 entries, 2022-01-03 to 2026-06-30
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   000001.ss  1084 non-null   float64
 1   bz=f       1128 non-null   float64
 2   eurusd=x   1167 non-null   float64
 3   gbpusd=x   1167 non-null   float64
 4   gc=f       1127 non-null   float64
 5   gd=f       1126 non-null   float64
 6   usdchf=x   1167 non-null   float64
 7   usdjpy=x   1167 non-null   float64
 8   xagg.to    1126 non-null   float64
 9   ^ftse      1131 non-null   float64
 10  ^gspc      1125 non-null   float64
 11  ^hsi       1099 non-null   float64
 12  ^ks11      1095 non-null   float64
 13  ^n225      1097 non-null   float64
 14  ^stoxx50e  1125 non-null   float64
dtypes: float64(15)
memory usage: 163.1 KB


In [ ]:
# obtain the total missing values for each variable
Total = df.isnull().sum().sort_values(ascending=False)
# the variable with highest percentage of missing values will appear first
Percent = (df.isnull().sum()*100/df.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([Total, Percent], axis = 1, keys = ['Total', 'Percentage of Missing Values'])
missing_data.head(35)

,Total,Percentage of Missing Values
000001.ss,221,16.934866
^ks11,210,16.091954
^n225,208,15.938697
^hsi,206,15.785441
^gspc,180,13.793103
^stoxx50e,180,13.793103
gd=f,179,13.716475
xagg.to,179,13.716475
gc=f,178,13.639847
bz=f,177,13.563218


In [ ]:
#check minimum and maximum date values

min(df.index), max(df.index)

(Timestamp('2022-01-03 00:00:00'), Timestamp('2026-06-30 00:00:00'))

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
000001.ss,1084.000000,3346.017630,362.425281,2702.185059,3086.893188,3263.584961,3457.299194,4242.571777
bz=f,1128.000000,82.881028,13.914254,58.919998,73.049997,81.385002,90.397497,127.980003
eurusd=x,1167.000000,1.095592,0.050324,0.959619,1.065252,1.088187,1.137896,1.201764
gbpusd=x,1167.000000,1.277678,0.058365,1.072754,1.241835,1.273000,1.331506,1.382533
gc=f,1127.000000,2645.108425,978.660541,1623.300049,1914.900024,2261.000000,3302.050049,5318.399902
gd=f,1126.000000,598.189014,69.743361,504.950012,548.625000,574.549988,620.935028,834.500000
usdchf=x,1167.000000,0.879825,0.059719,0.763200,0.823685,0.889390,0.919225,1.013420
usdjpy=x,1167.000000,144.891412,11.124174,113.785004,138.314499,147.192993,153.197998,161.923004
xagg.to,1126.000000,37.099241,1.112367,34.740002,36.169998,37.049999,37.970001,39.820000
^ftse,1131.000000,8268.380021,983.992887,6826.200195,7515.750000,7935.100098,8758.500000,10910.599609


In [ ]:
# Check if there are any duplicate rows
df.duplicated(keep=False).sum()

np.int64(0)

### Backfilling Missing Values using Forward-Fill

We will use the `ffill()` method to fill the `NaN` values. This method propagates the last valid observation forward to next valid observation. This is a common and appropriate technique for time series data as it ensures that we are not using future information to impute past values.

In [ ]:
# Apply forward-fill to the entire DataFrame
df_filled = df.ffill()

# Display the missing values after forward-filling
print("Missing values after forward-fill:")
Total_filled = df_filled.isnull().sum().sort_values(ascending=False)
Percent_filled = (df_filled.isnull().sum()*100/df_filled.isnull().count()).sort_values(ascending=False)
missing_data_filled = pd.concat([Total_filled, Percent_filled], axis = 1, keys = ['Total', 'Percentage of Missing Values'])
display(missing_data_filled.head(35))

# Display the head of the filled DataFrame to show the effect
print("\nHead of DataFrame after forward-fill:")
display(df_filled.head())

Missing values after forward-fill:


,Total,Percentage of Missing Values
000001.ss,1,0.076628
^ftse,1,0.076628
xagg.to,1,0.076628
^ks11,1,0.076628
^n225,1,0.076628
bz=f,0,0.000000
eurusd=x,0,0.000000
usdchf=x,0,0.000000
gd=f,0,0.000000
gc=f,0,0.000000



Head of DataFrame after forward-fill:


,000001.ss,bz=f,eurusd=x,gbpusd=x,gc=f,gd=f,usdchf=x,usdjpy=x,xagg.to,^ftse,^gspc,^hsi,^ks11,^n225,^stoxx50e
date,,,,,,,,,,,,,,,
2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078


You can see that most of the `NaN` values have been filled. Any remaining `NaN` values would typically only exist at the very beginning of a series if the first few entries were already `NaN` and there was no prior value to forward-fill from. In this case, there are no remaining `NaN` values as `ffill` successfully propagated values from the earliest available data points.

In [ ]:
df_filled.tail(2)

,000001.ss,bz=f,eurusd=x,gbpusd=x,gc=f,gd=f,usdchf=x,usdjpy=x,xagg.to,^ftse,^gspc,^hsi,^ks11,^n225,^stoxx50e
date,,,,,,,,,,,,,,,
2026-06-29,4073.902100,73.150002,1.142204,1.325399,4022.300049,618.950012,0.807600,161.923004,38.840000,10484.200195,7440.430176,23026.679688,8394.650391,69468.109375,6231.629883
2026-06-30,4073.902100,73.150002,1.142204,1.325399,4022.300049,618.950012,0.807600,161.923004,38.840000,10484.200195,7440.430176,23026.679688,8476.480469,70062.320312,6231.629883


In [ ]:
df_filled = df_filled.rename(columns={
    "000001.ss": "CN_Equity_SSE",
    "bz=f" : "BRENT_CRUDE_FUT",
    "eurusd=x": "EURUSD",
    "gbpusd=x": "GBPUSD",
    "gc=f": "GOLD_FUT",
    "gd=f": "GSCI_FUT",
    "usdchf=x": "USDCHF",
    "usdjpy=x": "USDJPY",
    "xagg.to":"US_BOND_AGG",
    "^ftse" : "UK_Equity_FTSE100",
    "^gspc": "US_Equity_SP500",
    "^hsi": "HK_Equity_HSI",
    "^ks11": "KR_Equity_KS11",
    "^n225": "JP_Equity_Nikkei225",
    "^stoxx50e": "EU_Equity_STOXX50E"
})

In [ ]:
df_filled.head()

,CN_Equity_SSE,BRENT_CRUDE_FUT,EURUSD,GBPUSD,GOLD_FUT,GSCI_FUT,USDCHF,USDJPY,US_BOND_AGG,UK_Equity_FTSE100,US_Equity_SP500,HK_Equity_HSI,KR_Equity_KS11,JP_Equity_Nikkei225,EU_Equity_STOXX50E
date,,,,,,,,,,,,,,,
2022-01-03,NaN,78.980003,1.137346,1.352228,1799.400024,563.900024,0.911975,115.141998,NaN,NaN,4796.560059,23274.750000,NaN,NaN,4331.819824
2022-01-04,3632.330078,80.000000,1.130224,1.348327,1814.000000,572.450012,0.918330,115.328003,39.279999,7505.200195,4793.540039,23289.839844,2989.239990,29301.789062,4367.620117
2022-01-05,3595.179932,80.800003,1.128363,1.353143,1824.599976,574.799988,0.916210,116.174004,39.279999,7516.899902,4700.580078,22907.250000,2953.969971,29332.160156,4392.149902
2022-01-06,3586.080078,81.989998,1.131350,1.355565,1788.699951,579.450012,0.917390,116.127998,39.279999,7450.399902,4696.049805,23072.859375,2920.530029,28487.869141,4324.810059
2022-01-07,3579.540039,81.750000,1.129688,1.353363,1797.000000,579.650024,0.921400,115.864998,39.279999,7485.299805,4677.029785,23493.380859,2954.889893,28478.560547,4305.830078


In [ ]:
#We reset the time series to only include modelable data (1971-2022)
df_filled = df_filled.loc['2022-01-04':'2026-06-30']

In [ ]:
#check minimum and maximum date values

min(df_filled.index), max(df_filled.index)

(Timestamp('2022-01-04 00:00:00'), Timestamp('2026-06-30 00:00:00'))

In [ ]:
# obtain the total missing values for each variable
Total = df_filled.isnull().sum().sort_values(ascending=False)
# the variable with highest percentage of missing values will appear first
Percent = (df_filled.isnull().sum()*100/df_filled.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([Total, Percent], axis = 1, keys = ['Total', 'Percentage of Missing Values'])
missing_data.head(35)

,Total,Percentage of Missing Values
CN_Equity_SSE,0,0.000000
BRENT_CRUDE_FUT,0,0.000000
EURUSD,0,0.000000
GBPUSD,0,0.000000
GOLD_FUT,0,0.000000
GSCI_FUT,0,0.000000
USDCHF,0,0.000000
USDJPY,0,0.000000
US_BOND_AGG,0,0.000000
UK_Equity_FTSE100,0,0.000000


In [ ]:
#from ydata_profiling import ProfileReport

In [ ]:
#profile = ProfileReport(df_filled, title="Data Profiling Report", explorative = True)

In [ ]:
#profile.to_notebook_iframe()

In [ ]:
log_returns = np.log(df_filled/df_filled.shift(1)).dropna()
print("\nLog Returns computed Successfully")
print(log_returns.head())


Log Returns computed Successfully
            CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD    GBPUSD  GOLD_FUT  \
date                                                                       
2022-01-05      -0.010280         0.009950 -0.001649  0.003566  0.005826   
2022-01-06      -0.002534         0.014620  0.002644  0.001788 -0.019872   
2022-01-07      -0.001825        -0.002931 -0.001470 -0.001625  0.004630   
2022-01-10       0.003898        -0.010823  0.005040  0.004327  0.000779   
2022-01-11      -0.007284         0.034635 -0.002076 -0.000883  0.011170   

            GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
date                                                                       
2022-01-05  0.004097 -0.002311  0.007309     0.000000           0.001558   
2022-01-06  0.008057  0.001287 -0.000396     0.000000          -0.008886   
2022-01-07  0.000345  0.004362 -0.002267     0.000000           0.004673   
2022-01-10 -0.007098 -0.002782 -0.002134     0.00000

In [ ]:
# Volatility and Correlation Structure

daily_volatility = log_returns.std()
correlation_matrix = log_returns.corr()

print("\nDaily Voltility Estimates: \n" , daily_volatility)
print("\nCorrelation matrix: \n" , correlation_matrix)


Daily Voltility Estimates: 
 CN_Equity_SSE         0.009211
BRENT_CRUDE_FUT       0.022899
EURUSD                0.004618
GBPUSD                0.005147
GOLD_FUT              0.011198
GSCI_FUT              0.013370
USDCHF                0.004900
USDJPY                0.006018
US_BOND_AGG           0.003960
UK_Equity_FTSE100     0.007465
US_Equity_SP500       0.010202
HK_Equity_HSI         0.014976
KR_Equity_KS11        0.015109
JP_Equity_Nikkei225   0.013460
EU_Equity_STOXX50E    0.010315
dtype: float64

Correlation matrix: 
                      CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD    GBPUSD  \
CN_Equity_SSE             1.000000         0.046532  0.111950  0.136424   
BRENT_CRUDE_FUT           0.046532         1.000000 -0.028638  0.002190   
EURUSD                    0.111950        -0.028638  1.000000  0.787601   
GBPUSD                    0.136424         0.002190  0.787601  1.000000   
GOLD_FUT                  0.138728         0.070028  0.225382  0.196187   
GSCI_FUT         

In [ ]:
correlation_matrix.to_csv("correlation_matrix.csv")
print("Correlation Matrix Saved Successfully")

Correlation Matrix Saved Successfully


In [ ]:
# Export Returns Matrix for VaR and ES

log_returns.to_csv("Market_RiskFactor_Returns.csv")

print("\nExport Created SUccsessfully:")
print(" - Market_RiskFactor_Returns.csv")


Export Created SUccsessfully:
 - Market_RiskFactor_Returns.csv


In [ ]:
print("Module 2.2 Started : Trading Portfolio Construction")

# load Risk Factor returns generated in previous Module (2.1)
returns = pd.read_csv("/content/Market_RiskFactor_Returns.csv" , index_col= 0, parse_dates= True)

Module 2.2 Started : Trading Portfolio Construction


In [ ]:
returns.head()

,CN_Equity_SSE,BRENT_CRUDE_FUT,EURUSD,GBPUSD,GOLD_FUT,GSCI_FUT,USDCHF,USDJPY,US_BOND_AGG,UK_Equity_FTSE100,US_Equity_SP500,HK_Equity_HSI,KR_Equity_KS11,JP_Equity_Nikkei225,EU_Equity_STOXX50E
date,,,,,,,,,,,,,,,
2022-01-05,-0.010280,0.009950,-0.001649,0.003566,0.005826,0.004097,-0.002311,0.007309,0.000000,0.001558,-0.019583,-0.016564,-0.011869,0.001036,0.005601
2022-01-06,-0.002534,0.014620,0.002644,0.001788,-0.019872,0.008057,0.001287,-0.000396,0.000000,-0.008886,-0.000964,0.007204,-0.011385,-0.029206,-0.015451
2022-01-07,-0.001825,-0.002931,-0.001470,-0.001625,0.004630,0.000345,0.004362,-0.002267,0.000000,0.004673,-0.004058,0.018062,0.011696,-0.000327,-0.004398
2022-01-10,0.003898,-0.010823,0.005040,0.004327,0.000779,-0.007098,-0.002782,-0.002134,0.000000,-0.005358,-0.001442,0.010718,-0.009579,0.000000,-0.015520
2022-01-11,-0.007284,0.034635,-0.002076,-0.000883,0.011170,0.023355,0.009057,-0.002945,0.000000,0.006173,0.009118,-0.000315,0.000225,-0.009033,0.009863


In [ ]:
  #Basel FRTB requires explicit risk factor categoriztion
  frtb_mapping = {
      "US_Equity_SP500": "Equity Risk Factor",
      "EU_Equity_STOXX50E": "Equity Risk Factor",
      "CN_Equity_SSE": "Equity Risk Factor",
      "JP_Equity_Nikkei225": "Equity Risk Factor",
      "KR_Equity_KS11": "Equity Risk Factor",
      "HK_Equity_HSI": "Equity Risk Factor",
      "UK_Equity_FTSE100":"Equity Risk Factor",
      "US_BOND_AGG":"Bond Risk Factor",
      "EURUSD":"FX Risk Factor",
      "GBPUSD":"FX Risk Factor",
      "USDJPY":"FX Risk Factor",
      "USDCHF":"FX Risk Factor",
      "GOLD_FUT":"Commodity Risk Factor",
      "GSCI_FUT":"Commodity Risk Factor",
      "BRENT_CRUDE_FUT":"Commodity Risk Factor"
  }

  mapping_df = pd.DataFrame.from_dict(
      frtb_mapping,
      orient="index",
      columns=["Risk Factor Category"]
  )

  print("\nBasel FRTB Risk Factor Classification")
  print(mapping_df)


Basel FRTB Risk Factor Classification
                      Risk Factor Category
US_Equity_SP500         Equity Risk Factor
EU_Equity_STOXX50E      Equity Risk Factor
CN_Equity_SSE           Equity Risk Factor
JP_Equity_Nikkei225     Equity Risk Factor
KR_Equity_KS11          Equity Risk Factor
HK_Equity_HSI           Equity Risk Factor
UK_Equity_FTSE100       Equity Risk Factor
US_BOND_AGG               Bond Risk Factor
EURUSD                      FX Risk Factor
GBPUSD                      FX Risk Factor
USDJPY                      FX Risk Factor
USDCHF                      FX Risk Factor
GOLD_FUT             Commodity Risk Factor
GSCI_FUT             Commodity Risk Factor
BRENT_CRUDE_FUT      Commodity Risk Factor


In [ ]:
WEIGHTS = {
    # Equity index book - 45% net, weighted toward US, then Europe/Japan/UK
    "US_Equity_SP500":    0.28, # Adjusted from 0.29 to make sum of weights 1.0
    "EU_Equity_STOXX50E":   0.06,
    "CN_Equity_SSE":    0.04,
    "JP_Equity_Nikkei225":    0.03,
    "KR_Equity_KS11":  .03,
    "HK_Equity_HSI":   .03,
    "UK_Equity_FTSE100":   .03,

    # Rates / government bond book - 25% net, long duration hedge
    "US_BOND_AGG":   0.25,

    # FX book - 20% gross notional, long EUR/GBP vs USD, short JPY/CHF vs USD
    # (small net long "risk" currencies, short "funding/safe-haven" currencies -
    # a classic G10 carry-style tilt, net FX exposure = +10%)
    "EURUSD":       0.08,
    "GBPUSD":       0.06,
    "USDJPY":       -0.03,   # negative weight = short USDJPY (long JPY)
    "USDCHF":       -0.01,   # negative weight = short USDCHF (long CHF)

    # Commodity book - 15% net, energy + gold overweight vs broad index
    "BRENT_CRUDE_FUT":  0.07,
    "GOLD_FUT":    0.06,
    "GSCI_FUT":   0.02,
}

weights = pd.Series(WEIGHTS).reindex(returns.columns).fillna(0.0)
assert abs(weights.sum() - 1.0) < 1e-9, f"Weights must sum to 1.0, got {weights.sum()}"

gross_exposure = weights.abs().sum()
net_exposure = weights.sum()

# ----------------------------------------------------------------------------
# 2. PORTFOLIO DAILY RETURNS (fixed-weight, daily-rebalanced)
# ----------------------------------------------------------------------------
portfolio_returns = returns.mul(weights, axis=1).sum(axis=1)
portfolio_returns.name = "PORTFOLIO_RETURN"

# Per-factor contribution to portfolio return each day (for P&L attribution)
contributions = returns.mul(weights, axis=1)
contributions.columns = [f"{c}_CONTRIB" for c in contributions.columns]

# ----------------------------------------------------------------------------
# 3. PORTFOLIO VALUE PATH (assume $100mm notional trading book)
# ----------------------------------------------------------------------------
NOTIONAL = 100_000_000
portfolio_value = NOTIONAL * (1 + portfolio_returns).cumprod()
portfolio_value.name = "PORTFOLIO_VALUE"

# ----------------------------------------------------------------------------
# 4. RISK METRICS - ann. vol, historical 1-day 99% VaR & ES (FRTB style)
# ----------------------------------------------------------------------------
ann_vol = portfolio_returns.std() * np.sqrt(252)
var_99 = -np.percentile(portfolio_returns, 1) * NOTIONAL
es_975 = -portfolio_returns[portfolio_returns <= np.percentile(portfolio_returns, 2.5)].mean() * NOTIONAL

asset_class_weights = {
    "Equity Index": weights[["US_Equity_SP500", "EU_Equity_STOXX50E", "CN_Equity_SSE", "JP_Equity_Nikkei225","KR_Equity_KS11" ,"HK_Equity_HSI" , "UK_Equity_FTSE100" ]].sum(),
    "Rates / Bond Index": weights[["US_BOND_AGG"]].sum(),
    "FX": weights[["EURUSD", "GBPUSD", "USDJPY", "USDCHF"]].sum(),
    "FX (gross)": weights[["EURUSD", "GBPUSD", "USDJPY", "USDCHF"]].abs().sum(),
    "Commodity": weights[["BRENT_CRUDE_FUT", "GOLD_FUT", "GSCI_FUT"]].sum(),
}

# ----------------------------------------------------------------------------
# 5. OUTPUT
# ----------------------------------------------------------------------------
weights.to_frame("weight").to_csv("portfolio_weights.csv")
portfolio_returns.to_csv("portfolio_daily_returns.csv")
portfolio_value.to_csv("portfolio_value_path.csv")
contributions.to_csv("portfolio_factor_contributions.csv")

print("Weights (net, sum = 1.0):")
print(weights)
print("\nGross exposure:", round(gross_exposure, 3), " Net exposure:", round(net_exposure, 3))
print("\nAsset-class net weights:")
for k, v in asset_class_weights.items():
    print(f"  {k}: {v:.2%}")
print("\nAnnualized portfolio volatility: {:.2%}".format(ann_vol))
print(f"1-day 99% historical VaR on ${NOTIONAL:,.0f} notional: ${var_99:,.0f}")
print(f"1-day 97.5% historical ES  on ${NOTIONAL:,.0f} notional: ${es_975:,.0f}")
print("\nFinal portfolio value:", f"${portfolio_value.iloc[-1]:,.0f}")

Weights (net, sum = 1.0):
CN_Equity_SSE          0.040000
BRENT_CRUDE_FUT        0.070000
EURUSD                 0.080000
GBPUSD                 0.060000
GOLD_FUT               0.060000
GSCI_FUT               0.020000
USDCHF                -0.010000
USDJPY                -0.030000
US_BOND_AGG            0.250000
UK_Equity_FTSE100      0.030000
US_Equity_SP500        0.280000
HK_Equity_HSI          0.030000
KR_Equity_KS11         0.030000
JP_Equity_Nikkei225    0.030000
EU_Equity_STOXX50E     0.060000
dtype: float64

Gross exposure: 1.08  Net exposure: 1.0

Asset-class net weights:
  Equity Index: 50.00%
  Rates / Bond Index: 25.00%
  FX: 10.00%
  FX (gross): 18.00%
  Commodity: 15.00%

Annualized portfolio volatility: 7.31%
1-day 99% historical VaR on $100,000,000 notional: $1,277,700
1-day 97.5% historical ES  on $100,000,000 notional: $1,366,870

Final portfolio value: $126,296,139


In [ ]:
# Volatility Modeling and Covariance Estimation for Parametric VaR
print ("Module 2.3 Started; Volatility + Covariance Estimation")

#load risk facor returns
returns = pd.read_csv("/content/Market_RiskFactor_Returns.csv" , index_col= 0, parse_dates= True)

#load Portfolio returns
portfolio_returns = pd.read_csv("/content/portfolio_daily_returns.csv" , index_col=0, parse_dates= True)

#load weights
#weights = pd.read_csv("/content/portfolio_weights.

print ("Datasets loaded successfully")
print("\nRisk Factor Returns Preview:")
print (returns.head())

print ("\nPortfolio Returns preview:")
print (portfolio_returns.head())

Module 2.3 Started; Volatility + Covariance Estimation
Datasets loaded successfully

Risk Factor Returns Preview:
            CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD    GBPUSD  GOLD_FUT  \
date                                                                       
2022-01-05      -0.010280         0.009950 -0.001649  0.003566  0.005826   
2022-01-06      -0.002534         0.014620  0.002644  0.001788 -0.019872   
2022-01-07      -0.001825        -0.002931 -0.001470 -0.001625  0.004630   
2022-01-10       0.003898        -0.010823  0.005040  0.004327  0.000779   
2022-01-11      -0.007284         0.034635 -0.002076 -0.000883  0.011170   

            GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
date                                                                       
2022-01-05  0.004097 -0.002311  0.007309     0.000000           0.001558   
2022-01-06  0.008057  0.001287 -0.000396     0.000000          -0.008886   
2022-01-07  0.000345  0.004362 -0.002267     0.00

In [ ]:
#Daily Volatility Estimation
daily_volatility = returns.std()
print ("\nDaily Volatility by Risk Factor:")
print (daily_volatility)

#banks typically annualize volatility using sqrt(252)
annualized_volatility = daily_volatility * np.sqrt(252)
print ("\nAnnualized Volatility by Risk Factor:")
print (annualized_volatility)


Daily Volatility by Risk Factor:
CN_Equity_SSE         0.009211
BRENT_CRUDE_FUT       0.022899
EURUSD                0.004618
GBPUSD                0.005147
GOLD_FUT              0.011198
GSCI_FUT              0.013370
USDCHF                0.004900
USDJPY                0.006018
US_BOND_AGG           0.003960
UK_Equity_FTSE100     0.007465
US_Equity_SP500       0.010202
HK_Equity_HSI         0.014976
KR_Equity_KS11        0.015109
JP_Equity_Nikkei225   0.013460
EU_Equity_STOXX50E    0.010315
dtype: float64

Annualized Volatility by Risk Factor:
CN_Equity_SSE         0.146216
BRENT_CRUDE_FUT       0.363503
EURUSD                0.073310
GBPUSD                0.081699
GOLD_FUT              0.177760
GSCI_FUT              0.212237
USDCHF                0.077789
USDJPY                0.095529
US_BOND_AGG           0.062860
UK_Equity_FTSE100     0.118506
US_Equity_SP500       0.161954
HK_Equity_HSI         0.237735
KR_Equity_KS11        0.239844
JP_Equity_Nikkei225   0.213668
EU_Equity_STO

In [ ]:
#Rolling Volatility Estimation
#Market Volatility is time varying, so rolling windows are used

rolling_window = 60
rolling_vol = returns.rolling(window=rolling_window).std()

print ("\nRolling Volatility Computed(60 day window)")
print (rolling_vol.tail(rolling_window))



Rolling Volatility Computed(60 day window)
            CN_Equity_SSE  BRENT_CRUDE_FUT   EURUSD   GBPUSD  GOLD_FUT  \
date                                                                     
2026-04-22       0.009265         0.049852 0.004433 0.004878  0.019020   
2026-04-23       0.009210         0.049847 0.004434 0.004880  0.019024   
2026-04-24       0.009182         0.049642 0.004432 0.004857  0.018974   
2026-04-26       0.009176         0.049651 0.004433 0.004739  0.018813   
2026-04-27       0.008993         0.049709 0.004405 0.004667  0.018650   
...                   ...              ...      ...      ...       ...   
2026-06-25       0.007410         0.032399 0.002630 0.003502  0.013959   
2026-06-26       0.007987         0.030615 0.002630 0.003502  0.013898   
2026-06-28       0.007987         0.030615 0.002625 0.003487  0.013898   
2026-06-29       0.008067         0.029751 0.002613 0.003504  0.013936   
2026-06-30       0.008067         0.029393 0.002586 0.003500  0.0136

In [ ]:
# Covariance Matrix estimation
# covariance captures joint movements across risk factors
cov_matrix = returns.cov()
print ("\nCovariance Matrix (Daily):")
print (cov_matrix)

#Correlation Matrix
# Correlation is covariance normalized into a [-1,1]scale
corr_matrix = returns.corr()
print ("\nCorrelation Matrix (Daily):")
print (corr_matrix)

#Annualize


Covariance Matrix (Daily):
                     CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD    GBPUSD  \
CN_Equity_SSE             0.000085         0.000010  0.000005  0.000006   
BRENT_CRUDE_FUT           0.000010         0.000524 -0.000003  0.000000   
EURUSD                    0.000005        -0.000003  0.000021  0.000019   
GBPUSD                    0.000006         0.000000  0.000019  0.000026   
GOLD_FUT                  0.000014         0.000018  0.000012  0.000011   
GSCI_FUT                  0.000010         0.000267 -0.000000  0.000003   
USDCHF                   -0.000004         0.000005 -0.000017 -0.000016   
USDJPY                   -0.000003         0.000003 -0.000014 -0.000015   
US_BOND_AGG              -0.000000        -0.000007 -0.000002 -0.000001   
UK_Equity_FTSE100         0.000011         0.000007  0.000002  0.000001   
US_Equity_SP500           0.000007         0.000008  0.000004  0.000006   
HK_Equity_HSI             0.000080         0.000023  0.000012  0.000012 

In [ ]:
# Portolio volatility using Covariance
# weights same as previous

# Portfilo Variance
portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
#print ("\nPortfolio Variance (Covariance):")
#print (portfolio_variance)
portfolio_volatility = np.sqrt(portfolio_variance)
print ("\nPortfolio Volatility (Daily, Covariance-Based):")
print (portfolio_volatility)

#Export Parametric VaR inputs
cov_matrix.to_csv("covariance_matrix.csv")
print("\nCovariance Matrix Saved Successfully")


Portfolio Volatility (Daily, Covariance-Based):
0.004605246385086062

Covariance Matrix Saved Successfully


In [ ]:
# Return Diagnostics
#Skewness and Kurtosis
# Skewness measures asymmetry,
# Kurtosis measures tail heaviness

skewness = returns.skew()
kurtosis = returns.kurtosis()
print("\nSkewness of Risk facor returns:")
print (skewness)
print ("\nKurtosis of Risk factor returns:")
print (kurtosis)





Skewness of Risk facor returns:
CN_Equity_SSE         -0.441267
BRENT_CRUDE_FUT       -0.957092
EURUSD                 0.225364
GBPUSD                -0.223315
GOLD_FUT              -1.255357
GSCI_FUT              -0.657795
USDCHF                -0.892652
USDJPY                -0.494665
US_BOND_AGG            0.250266
UK_Equity_FTSE100     -0.740602
US_Equity_SP500        0.056823
HK_Equity_HSI         -0.358012
KR_Equity_KS11        -0.691022
JP_Equity_Nikkei225   -0.408689
EU_Equity_STOXX50E    -0.105748
dtype: float64

Kurtosis of Risk factor returns:
CN_Equity_SSE         12.309132
BRENT_CRUDE_FUT        6.014530
EURUSD                 2.422344
GBPUSD                 5.931717
GOLD_FUT              13.182676
GSCI_FUT               4.501676
USDCHF                 5.898481
USDJPY                 3.407869
US_BOND_AGG           10.082351
UK_Equity_FTSE100      6.216448
US_Equity_SP500        7.818633
HK_Equity_HSI          9.684940
KR_Equity_KS11        12.763442
JP_Equity_Nikkei225   

In [ ]:
#Portfolio Returns Distribution Summary

portfolio_mean = portfolio_returns.mean().values[0]
portfolio_std = portfolio_returns.std().values[0]
print("\nPortfolio Mean Return", portfolio_mean)
print("\nPortfolio Volatility", portfolio_std)

# Extreme Quantiles (tail risk)
q_1 = portfolio_returns.quantile(0.01).values[0]
q_5 = portfolio_returns.quantile(0.05).values[0]

print("\n1% Extreme Quantile", q_1)
print("\n5% Extreme Quantile", q_5)




Portfolio Mean Return 0.00018979634477815614

Portfolio Volatility 0.0046052463850860285

1% Extreme Quantile -0.012776996938915052

5% Extreme Quantile -0.00742772930930927


In [ ]:
#Identify Extreme Stress Days
threshold = portfolio_returns.quantile(.01).values[0]

stress_days = portfolio_returns[portfolio_returns["PORTFOLIO_RETURN"] <= threshold]
print("\nNumber of Extreme Stress Days (Bottom 1%):" , stress_days.shape[0])
print("\nSample Stress Events:")
print(stress_days.head(20))



Number of Extreme Stress Days (Bottom 1%): 14

Sample Stress Events:
            PORTFOLIO_RETURN
date                        
2022-04-06         -0.013244
2022-05-09         -0.017685
2022-05-18         -0.015520
2022-06-13         -0.019057
2022-09-13         -0.016309
2022-09-23         -0.013278
2023-08-02         -0.012787
2024-08-02         -0.013195
2024-08-05         -0.018796
2025-04-03         -0.025522
2025-04-04         -0.027289
2025-04-07         -0.024116
2026-03-23         -0.014006
2026-06-05         -0.014380


In [ ]:
#add Stress Flag to dataset
portfolio_returns["Stress Flag"] = (
    portfolio_returns["PORTFOLIO_RETURN"] <= threshold
).astype(int)

print("\nStress Flag Added (1 = Extreme Stress Day)")
print(portfolio_returns.head(10))


Stress Flag Added (1 = Extreme Stress Day)
            PORTFOLIO_RETURN  Stress Flag
date                                     
2022-01-05         -0.005320            0
2022-01-06         -0.002257            0
2022-01-07         -0.000561            0
2022-01-10         -0.001404            0
2022-01-11          0.006105            0
2022-01-12          0.005563            0
2022-01-13         -0.009466            0
2022-01-14         -0.000026            0
2022-01-17          0.000022            0
2022-01-18         -0.005069            0


In [ ]:
#Final VaR Model Imput Pack Export

#combine porfolio returns + stress indicator
portfolio_returns.to_csv("VaR Model Imput Pack.csv")
print("\nVaR Model Imput Pack Saved Successfully")
print(" - VaR_Model_Imput_Pack.csv")


VaR Model Imput Pack Saved Successfully
 - VaR_Model_Imput_Pack.csv


In [ ]:
#Parametric VaR Engine
#load Portfolio Daily Returns
portfolio_returns = pd.read_csv("/content/portfolio_daily_returns.csv" , index_col= 0, parse_dates= True)

#Load Covariance Matrix
cov_matrix = pd.read_csv("/content/covariance_matrix.csv" , index_col= 0)

#load weights
weights = pd.read_csv("/content/portfolio_weights.csv" , index_col= 0)

print("\nPortfolio Returns preview")
print(portfolio_returns.head())

print("\nCovariance Matrix preview")
print(cov_matrix.head())

print("\nWeights preview")
print(weights.head())




Portfolio Returns preview
            PORTFOLIO_RETURN
date                        
2022-01-05         -0.005320
2022-01-06         -0.002257
2022-01-07         -0.000561
2022-01-10         -0.001404
2022-01-11          0.006105

Covariance Matrix preview
                 CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD   GBPUSD  GOLD_FUT  \
CN_Equity_SSE         0.000085         0.000010  0.000005 0.000006  0.000014   
BRENT_CRUDE_FUT       0.000010         0.000524 -0.000003 0.000000  0.000018   
EURUSD                0.000005        -0.000003  0.000021 0.000019  0.000012   
GBPUSD                0.000006         0.000000  0.000019 0.000026  0.000011   
GOLD_FUT              0.000014         0.000018  0.000012 0.000011  0.000125   

                 GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
CN_Equity_SSE    0.000010 -0.000004 -0.000003    -0.000000           0.000011   
BRENT_CRUDE_FUT  0.000267  0.000005  0.000003    -0.000007           0.000007   
EURUSD          -0.

In [ ]:
from scipy.stats import norm

#Define Portfolio Structure and confidence level
Confidence_level = .99 #Basel VaR confidence level

z_score = norm.ppf(Confidence_level)
print("\nConfidence Level:", Confidence_level)
print("z-score (Normal Quantile):" , z_score)


Confidence Level: 0.99
z-score (Normal Quantile): 2.3263478740408408


In [ ]:
#Portfolio Volatility Computation
portfolio_variance = np.dot(weights.T, np.dot(cov_matrix.values, weights))
portfolio_volatility = np.sqrt(portfolio_variance)
print("\nPortfoio Variance:" , portfolio_variance)
print("Portfolio Volatility (Daily):", portfolio_volatility)



Portfoio Variance: [[2.12082943e-05]]
Portfolio Volatility (Daily): [[0.00460525]]


In [ ]:
#compute 1 day Parametric VaR
VaR_1day = z_score * portfolio_volatility
print("1-day Parametric VaR (1-Day, 99%):", VaR_1day)

1-day Parametric VaR (1-Day, 99%): [[0.01071341]]


In [ ]:
#Basel 10 day VaR Scaling
VaR_10day = VaR_1day * np.sqrt(10)
print("10-day Parametric VaR (10-Day Horizon):", VaR_10day)


10-day Parametric VaR (10-Day Horizon): [[0.03387876]]


In [ ]:
#VaR in Monetary Terms
portfolio_value = 100_000_000
VaR_1day_monetary = VaR_1day * portfolio_value
VaR_10day_monetary = VaR_10day * portfolio_value

print("1-day Parametric VaR (1-Day, 99%):", VaR_1day_monetary)
print("10-day Parametric VaR (10-Day Horizon):", VaR_10day_monetary)

1-day Parametric VaR (1-Day, 99%): [[1071340.51373775]]
10-day Parametric VaR (10-Day Horizon): [[3387876.17302619]]


In [ ]:
#Export Parametric VaR Report
VaR_report = pd.DataFrame ({
    "metric":["VaR_1day", "VaR_10day"],
    "VaR_percentage":[VaR_1day.item(), VaR_10day.item()],
    "Var_Monetary":[VaR_1day_monetary.item(), VaR_10day_monetary.item()]
})

VaR_report.to_csv("VaR_parametric_report.csv")

print("\nVaR Parametric Report Saved Successfully")

print(VaR_report)


VaR Parametric Report Saved Successfully
      metric  VaR_percentage   Var_Monetary
0   VaR_1day        0.010713 1071340.513738
1  VaR_10day        0.033879 3387876.173026


In [ ]:
# Historical Simulation VaR Engine

#Load Portfolio returns
portfolio_returns = pd.read_csv("/content/portfolio_daily_returns.csv" , index_col= 0, parse_dates= True)
print(portfolio_returns.head())

            PORTFOLIO_RETURN
date                        
2022-01-05         -0.005320
2022-01-06         -0.002257
2022-01-07         -0.000561
2022-01-10         -0.001404
2022-01-11          0.006105


In [ ]:
#Convert Returns into Losses
portfolio_returns["Portfolio_Loss"] = -portfolio_returns["PORTFOLIO_RETURN"]

print("\nPortfolio Loss Series Created:")
print(portfolio_returns.head())


Portfolio Loss Series Created:
            PORTFOLIO_RETURN  Portfolio_Loss
date                                        
2022-01-05         -0.005320        0.005320
2022-01-06         -0.002257        0.002257
2022-01-07         -0.000561        0.000561
2022-01-10         -0.001404        0.001404
2022-01-11          0.006105       -0.006105


In [ ]:
#Historical VaR Computation

Confidence_level = 0.99
historical_var = portfolio_returns["Portfolio_Loss"].quantile(1 - Confidence_level)

print("\nHistorical Simulation VaR (1-Day 99%):", historical_var)




Historical Simulation VaR (1-Day 99%): -0.011404250185555787


In [ ]:
#Rolling Historical VaR Series
#banks compute VaR daily using rolling historical windows

window = 250 # 1 year trading window
portfolio_returns["Rolling_Historical_VaR"] = (
    portfolio_returns["Portfolio_Loss"]
    .rolling(window)
    .quantile(Confidence_level)
)

print("\nRolling Historical VaR Series Computed")
print(portfolio_returns[["Portfolio_Loss" ,"Rolling_Historical_VaR"]].tail())


Rolling Historical VaR Series Computed
            Portfolio_Loss  Rolling_Historical_VaR
date                                              
2026-06-25       -0.005815                0.011574
2026-06-26        0.007632                0.011574
2026-06-28       -0.000219                0.011574
2026-06-29       -0.005554                0.011574
2026-06-30       -0.000547                0.011574


In [ ]:
# Identity VaR Breaches

portfolio_returns["VaR Breach"] = (
    portfolio_returns["Portfolio_Loss"] >
    portfolio_returns["Rolling_Historical_VaR"]
).astype(int)

breaches = portfolio_returns["VaR Breach"].sum()

print("\nTotal VaR Breaches Detected:", breaches)


Total VaR Breaches Detected: 12


In [ ]:
# Export Historical VaR  Results

output = portfolio_returns[[
    "PORTFOLIO_RETURN",
    "Portfolio_Loss",
    "Rolling_Historical_VaR",
    "VaR Breach",
]]

output.to_csv("Historical_VaR_Output.csv")

print("\nHistorical VaR Results Saved Successfully")
print(" - Historical_VaR_Output.csv")


Historical VaR Results Saved Successfully
 - Historical_VaR_Output.csv


In [ ]:
# Monte Carlo VaR simulation Engine module 3.3( Basel Portfolio Risk Modeling)

print("Monte Carlo VaR Engine")

#load covariace matrix important in MC VaR, ot preserves realistic correlations structures across risk factors for MC simulations
cov_matrix = pd.read_csv("/content/covariance_matrix.csv" , index_col= 0)

#load Portfolio Weights
weights = pd.read_csv("/content/portfolio_weights.csv" , index_col= 0)

print("\nCovariance Matrix preview")
print(cov_matrix.head())

print("\nWeights preview")
print(weights.head())


Monte Carlo VaR Engine

Covariance Matrix preview
                 CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD   GBPUSD  GOLD_FUT  \
CN_Equity_SSE         0.000085         0.000010  0.000005 0.000006  0.000014   
BRENT_CRUDE_FUT       0.000010         0.000524 -0.000003 0.000000  0.000018   
EURUSD                0.000005        -0.000003  0.000021 0.000019  0.000012   
GBPUSD                0.000006         0.000000  0.000019 0.000026  0.000011   
GOLD_FUT              0.000014         0.000018  0.000012 0.000011  0.000125   

                 GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
CN_Equity_SSE    0.000010 -0.000004 -0.000003    -0.000000           0.000011   
BRENT_CRUDE_FUT  0.000267  0.000005  0.000003    -0.000007           0.000007   
EURUSD          -0.000000 -0.000017 -0.000014    -0.000002           0.000002   
GBPUSD           0.000003 -0.000016 -0.000015    -0.000001           0.000001   
GOLD_FUT         0.000030 -0.000014 -0.000012    -0.000001      

In [ ]:
#Simulation Parameters
n_simulations = 10000
Confidence_level = 0.99

print("\nMonte Carlo Simulation Parameters")
print("Number of Simulations:", n_simulations)
print("Confidence Level:", Confidence_level)


Monte Carlo Simulation Parameters
Number of Simulations: 10000
Confidence Level: 0.99


In [ ]:
#Simulate Correlated Risk Factor Returns

mean_vector = np.zeros(len(weights))

simulated_returns = np.random.multivariate_normal(
    mean=mean_vector,
    cov = cov_matrix.values,
    size=n_simulations
)

print("\nSimulated Returns Matrix Shape:" , simulated_returns.shape )




Simulated Returns Matrix Shape: (10000, 15)


In [ ]:
#Simulated Portfilio Returns

portfolio_simulated_returns = simulated_returns.dot(weights).flatten()


print("\nSimulated Portfolio Returns Preview:")
print( portfolio_simulated_returns[:5])


Simulated Portfolio Returns Preview:
[ 0.00088543 -0.00527314  0.00101635 -0.00574677 -0.00784187]


In [ ]:
#Portfolio Loss Distribution
portfolio_losses = -portfolio_simulated_returns

print("\nSimulated Portfolio Loss preview:" , portfolio_losses[:5])


Simulated Portfolio Loss preview: [-0.00088543  0.00527314 -0.00101635  0.00574677  0.00784187]


In [ ]:
#Monte Carlo VaR Computation

monteCarlo_VaR = np.quantile(portfolio_losses, Confidence_level)

print("\nMonte Carlo VaR (1 Day, 99%):" , monteCarlo_VaR)


Monte Carlo VaR (1 Day, 99%): 0.010606272833254772


In [ ]:
#Monte Carlo VaR in Money Terms

portfolio_value = 100_000_000
monteCarlo_VaR_monetary = monteCarlo_VaR * portfolio_value

print("\nMonte Carlo VaR (1 Day, 99%):" , monteCarlo_VaR_monetary)
#


Monte Carlo VaR (1 Day, 99%): 1060627.2833254773


In [ ]:
#Export Monte Carlo VaR Output

mc_report = pd.DataFrame({
    "Metric":["MonteCarlo_VaR_ 1Day_99%"],
    "VaR_Percentage":[monteCarlo_VaR],
    "VaR_Monetary":[monteCarlo_VaR_monetary],
    "Simulations" : [n_simulations]
})

mc_report.to_csv("Monte_Carlo_VaR_Output.csv" , index=False)

print("\nMonte Carlo VaR Output Saved Successfully")
print(" - Monte_Carlo_VaR_Output.csv")

print("\nMonte Carlo VaR Report:")
print(mc_report)


Monte Carlo VaR Output Saved Successfully
 - Monte_Carlo_VaR_Output.csv

Monte Carlo VaR Report:
                     Metric  VaR_Percentage   VaR_Monetary  Simulations
0  MonteCarlo_VaR_ 1Day_99%        0.010606 1060627.283325        10000


In [ ]:
# VaR Method Comparison + Management risk Dashbaord ( Basel Reporting Style)

print("VaR Comparison Dashboard")
# load results
parametric = pd.read_csv("/content/VaR_parametric_report.csv" , index_col=0)
historical = pd.read_csv("/content/Historical_VaR_Output.csv" , index_col= 0)
monteCarlo = pd.read_csv("/content/Monte_Carlo_VaR_Output.csv" , index_col= 0)

print("\nParametric VaR Report:")
print(parametric)



print("\nMonte Carlo VaR Report:")
print(monteCarlo)

VaR Comparison Dashboard

Parametric VaR Report:
      metric  VaR_percentage   Var_Monetary
0   VaR_1day        0.010713 1071340.513738
1  VaR_10day        0.033879 3387876.173026

Monte Carlo VaR Report:
                          VaR_Percentage   VaR_Monetary  Simulations
Metric                                                              
MonteCarlo_VaR_ 1Day_99%        0.010606 1060627.283325        10000


In [ ]:
#Consolidated VaR Summary Table
VaR_summary = pd.DataFrame({
    "method":["Parametric VaR" , "Historical VaR" , "Monte Carlo VaR"],
    "VaR_1day_99%":[
       parametric.loc[0, "VaR_percentage"],
       historical["Rolling_Historical_VaR"]. dropna().iloc[-1],
       monteCarlo.iloc[0]["VaR_Percentage"]
    ]

})

print("\nConsolidated VaR Summary Table:")
print(VaR_summary)
#


Consolidated VaR Summary Table:
            method  VaR_1day_99%
0   Parametric VaR      0.010713
1   Historical VaR      0.011574
2  Monte Carlo VaR      0.010606


In [ ]:
#VaR in Monetary Loss Terms
VaR_summary["VaR_Monetary_USD"] = VaR_summary["VaR_1day_99%"] * portfolio_value

print("\nVaR Summary Table with Monetary Impact:")
print(VaR_summary)


VaR Summary Table with Monetary Impact:
            method  VaR_1day_99%  VaR_Monetary_USD
0   Parametric VaR      0.010713    1071340.513738
1   Historical VaR      0.011574    1157447.670971
2  Monte Carlo VaR      0.010606    1060627.283325


In [ ]:
#Model Spread Analysis (benchmark vs baseline input)
#Used to detect model stability and unusual shifts in risk matrix

VaR_summary ["Difference_vs_Parametric"] = (
    VaR_summary["VaR_1day_99%"] - parametric.loc[0, "VaR_percentage"]
)

print("\nVaR Model Spread vs Parametric Benchmark:")
print(VaR_summary)
#


VaR Model Spread vs Parametric Benchmark:
            method  VaR_1day_99%  VaR_Monetary_USD  Difference_vs_Parametric
0   Parametric VaR      0.010713    1071340.513738                  0.000000
1   Historical VaR      0.011574    1157447.670971                  0.000861
2  Monte Carlo VaR      0.010606    1060627.283325                 -0.000107


In [ ]:
# Export VaR Dashboard Report
# management ready report
# used in daily trade risk reporting, risk committees
# used in Basel model risk reviews

VaR_summary.to_csv("VaR_Comparison_Dashboard.csv" , index=False)

print("\nVaR_Comparison_Dashboard Saved Successfully")
print(" - VaR_Comparison_Dashboard.csv")


VaR_Comparison_Dashboard Saved Successfully
 - VaR_Comparison_Dashboard.csv


In [ ]:
#Historical Expected Shortfall Engine (Basel FRTB 97.5 % Tail Risk Measure) Module 4.1

#foundation of FRTB Internal Models Approach

#load Historical VaR dataset
data = pd.read_csv("/content/Historical_VaR_Output.csv" , index_col= 0)

print("\nHistorical VaR Dataset:")

print(data.head())


Historical VaR Dataset:
            PORTFOLIO_RETURN  Portfolio_Loss  Rolling_Historical_VaR  \
date                                                                   
2022-01-05         -0.005320        0.005320                     NaN   
2022-01-06         -0.002257        0.002257                     NaN   
2022-01-07         -0.000561        0.000561                     NaN   
2022-01-10         -0.001404        0.001404                     NaN   
2022-01-11          0.006105       -0.006105                     NaN   

            VaR Breach  
date                    
2022-01-05           0  
2022-01-06           0  
2022-01-07           0  
2022-01-10           0  
2022-01-11           0  


In [ ]:
#FRTB Confidence Level

Confidence_level = 0.975
print("\nBasel FRTB Expetced Shortfall Confidence level:" , Confidence_level)


Basel FRTB Expetced Shortfall Confidence level: 0.975


In [ ]:
#Portfolio loss Series
# represents the realized trading book losses

losses = data["Portfolio_Loss"].dropna()
print("\nPortfolio Loss Series Summary:")
print(losses.describe())


Portfolio Loss Series Summary:
count   1303.000000
mean      -0.000190
std        0.004605
min       -0.024238
25%       -0.002663
50%       -0.000223
75%        0.001840
max        0.027289
Name: Portfolio_Loss, dtype: float64


In [ ]:
#compute Historical VaR Cutt-Off (97.5%)

VaR_cutoff = np.quantile(losses, Confidence_level)
print("\nHistorical VaR Cut-Off (97.5%):" , VaR_cutoff)


Historical VaR Cut-Off (97.5%): 0.009453314821472056


In [ ]:
# Compute Expected Shortfall ( Tail Average Loss)
# represents the worst tail outcomes
# in a crisis, ecpected magnitude of the loss # more coherent and conservatine than VaR

tail_losses = losses[losses >= VaR_cutoff]
expected_shortfall = tail_losses.mean()
print("\nHistorical Expected Shortfall (97.5):" , expected_shortfall)


Historical Expected Shortfall (97.5): 0.013668698694334165


In [ ]:
#Monte Carlo VaR Cutoff (97.5%)


VaR_cutoff = np.quantile(portfolio_losses, Confidence_level)
print("\nMonte Carlo VaR Cutoff(97.5%):" , VaR_cutoff)

#


Monte Carlo VaR Cutoff(97.5%): 0.008923202457311149


In [ ]:
# ES vs VaR Comparison
print("\nComparison of Tail Risk Measures:")
print("VaR cuttoff (9735%):" , VaR_cutoff)
print("Historical Expected Shortfall (97.5%):" , expected_shortfall)

#focus on how heavy the tail is
# directly impacts the capital requirements
severity_ratio = expected_shortfall / VaR_cutoff

print("\nTail Severity Ratios (ES/VaR)):" , severity_ratio)
#


Comparison of Tail Risk Measures:
VaR cuttoff (9735%): 0.008923202457311149
Historical Expected Shortfall (97.5%): 0.013668698694334165

Tail Severity Ratios (ES/VaR)): 1.5318153723089443


In [ ]:
# ES in Monetary Terms


portfolio_values = 100_000_000
ES_money = expected_shortfall * portfolio_value

print("\nExpected Shortfall in USD:")
print(ES_money)


Expected Shortfall in USD:
1366869.8694334165


In [ ]:
    #Export ES Output
    # Basel FRTB compliant tail risk deliverable
    # Foudation of stress calibration
    # Capital computation under internal model approach report file

es_report = pd.DataFrame({
    "Metric": ["Historical_ES_97.5%"],
    "ES_Percentage": [expected_shortfall],
    "ES_Monetary": [ES_money],
    "VaR_Cutoff": [VaR_cutoff],

})

es_report.to_csv("Historical_ES_Output.csv" , index=False)

print("\nHS Expected Shortfall Output Saved Successfully")
print(" - Historical_ES_Output.csv")
print(es_report)


HS Expected Shortfall Output Saved Successfully
 - Historical_ES_Output.csv
                Metric  ES_Percentage    ES_Monetary  VaR_Cutoff
0  Historical_ES_97.5%       0.013669 1366869.869433    0.008923


In [ ]:
#Monte Carlo Expected Shortfall
tail_losses = portfolio_losses[portfolio_losses >= VaR_cutoff]

MonteCarlo_ES = tail_losses.mean()

print("\nMonte Carlo Expected Shortfall:" , MonteCarlo_ES)


Monte Carlo Expected Shortfall: 0.010620370371917117


In [ ]:
#Tail Severity Analysis

severity_ratio = MonteCarlo_ES / VaR_cutoff

print("\nTail Severity Ratio (ES/VaR):" , severity_ratio)

print("\nTail Severity Ratio:" , severity_ratio)


Tail Severity Ratio (ES/VaR): 1.19019717671153

Tail Severity Ratio: 1.19019717671153


In [ ]:
# Monetary Expected shortfall

portfolio_value = 100_000_000 #USD trading book size

ES_money = MonteCarlo_ES * portfolio_value

print("\nMonte Carlo Expetected Shortfall in USD:" , ES_money)


Monte Carlo Expetected Shortfall in USD: 1062037.0371917118


In [ ]:
#Export Monte Carlo RS Report

mc_es_report = pd.DataFrame({
    "metric":["MonteCarlo_ES_97.5%"],
    "ES_Percentage" : [MonteCarlo_ES],
    "ES_Monetary":[ES_money],
    "VaR_Cutoff":[VaR_cutoff],
    "Simulations":[n_simulations]
})

mc_es_report.to_csv("MonteCarlo_ES_output.csv" , index = False)

print ("\nExport Created Successfully:")
print(" - MonteCarlo_ES_output.csv")

print ("\nMonte Carlo ES Summary:")
print(mc_es_report)

print("\nMonte Carlo ES Report Saved Successfully")
print(" - MonteCarlo_ES_report.csv")





Export Created Successfully:
 - MonteCarlo_ES_output.csv

Monte Carlo ES Summary:
                metric  ES_Percentage    ES_Monetary  VaR_Cutoff  Simulations
0  MonteCarlo_ES_97.5%       0.010620 1062037.037192    0.008923        10000

Monte Carlo ES Report Saved Successfully
 - MonteCarlo_ES_report.csv


In [ ]:
#4.2: Monte Carlo Expected Shortfall Engine
#Basel FRTB Simulation-Based tail Risk


#load covariance matrix # realistic covariance structures across the risk factors
cov_matrix = pd.read_csv("/content/covariance_matrix.csv" , index_col= 0)

#load Portfolio weights
weights = pd.read_csv("/content/portfolio_weights.csv" , index_col = 0)

print ("\nCovariance Matrix Preview:")
print(cov_matrix.head())






Covariance Matrix Preview:
                 CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD   GBPUSD  GOLD_FUT  \
CN_Equity_SSE         0.000085         0.000010  0.000005 0.000006  0.000014   
BRENT_CRUDE_FUT       0.000010         0.000524 -0.000003 0.000000  0.000018   
EURUSD                0.000005        -0.000003  0.000021 0.000019  0.000012   
GBPUSD                0.000006         0.000000  0.000019 0.000026  0.000011   
GOLD_FUT              0.000014         0.000018  0.000012 0.000011  0.000125   

                 GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
CN_Equity_SSE    0.000010 -0.000004 -0.000003    -0.000000           0.000011   
BRENT_CRUDE_FUT  0.000267  0.000005  0.000003    -0.000007           0.000007   
EURUSD          -0.000000 -0.000017 -0.000014    -0.000002           0.000002   
GBPUSD           0.000003 -0.000016 -0.000015    -0.000001           0.000001   
GOLD_FUT         0.000030 -0.000014 -0.000012    -0.000001           0.000010   

    

In [ ]:
#Basel FRTB Parameters

Confidence_level = 0.975 # Basel  FRTB ES Confidence level
n_simulations = 20_000   # High simulation for ES Stability

print ("\nBasel FRTB Monte Carlo ES Parameters:")
print("Confidence Level:" , Confidence_level)
print("Number of Simulations:" , n_simulations)
#


Basel FRTB Monte Carlo ES Parameters:
Confidence Level: 0.975
Number of Simulations: 20000


In [ ]:
# Simulate Correlated Risk Factor Returns
# Generate correlated market scenarios

mean_vector = np.zeros(len(weights))

simulated_returns = np.random.multivariate_normal(
    mean=mean_vector,
    cov=cov_matrix.values,
    size=n_simulations
)

print("\nSimulated Returns Matrix Shape:" , simulated_returns.shape)






Simulated Returns Matrix Shape: (20000, 15)


In [ ]:
# Portfolio returns and Loss Distribution

portfolio_simulated_returns = simulated_returns.dot(weights).flatten()

print("\nSimulated Portfolio Returns Preview:")
print(portfolio_simulated_returns[:5])

portfolio_losses = -portfolio_simulated_returns

print ("\nSimulated Portfolio Loss Preview:")
print(portfolio_losses[:5])
#


Simulated Portfolio Returns Preview:
[-0.0012119   0.00273834 -0.00031561  0.00349416  0.0064529 ]

Simulated Portfolio Loss Preview:
[ 0.0012119  -0.00273834  0.00031561 -0.00349416 -0.0064529 ]


In [ ]:
# Monte Carlo VaR Cutoff (97.5%)

VaR_cuttoff = np.quantile(portfolio_losses, Confidence_level)

print("\nMonte Carlo VaR Cutoff (97.5%):" , VaR_cuttoff)
#


Monte Carlo VaR Cutoff (97.5%): 0.009072136230364394


In [ ]:
# Monte Carlo Expected Shortfall

tail_losses = portfolio_losses[portfolio_losses >= VaR_cuttoff]

MonteCarlo_ES = tail_losses.mean()

print("\nMonte Carlo Expected Shortfall(97.5%):" , MonteCarlo_ES)



Monte Carlo Expected Shortfall(97.5%): 0.01077647762878027


In [ ]:
# Tail Severity Analysis
# In portfolios w heavy tails ES becomes sig. larger than VaR , crisis vulnerability

severity_ratio = MonteCarlo_ES / VaR_cuttoff

print("\nTail Severity Ratio (ES/VaR):" , severity_ratio)



Tail Severity Ratio (ES/VaR): 1.187865498834933


In [ ]:
# Monetary ES

portfolio_value = 100_000_000 # USD trading book size

ES_money = MonteCarlo_ES * portfolio_value

print("\nMonte Carlo Expected Shortfall in USD:" , ES_money)




Monte Carlo Expected Shortfall in USD: 1077647.762878027


In [ ]:
# Export Monte carlo ES Report
# output aligned w Basel Internal models Approach requirements

mc_es_report = pd.DataFrame({
    "Metric":["MonteCarlo_ES_97.5%"],
    "ES_Percentage":[MonteCarlo_ES],
    "ES_Monetary":[ES_money],
    "VaR_Cutoff":[VaR_cuttoff],
    "Simulations":[n_simulations]
})

mc_es_report.to_csv("MonteCarlo_ES_Output.csv" , index=False)

print("\nMonte Carlo ES Report Saved Successfully")
print(" - MonteCarlo_ES_Output.csv")

print("\nMonte Carlo ES Summary:")
print(mc_es_report)


Monte Carlo ES Report Saved Successfully
 - MonteCarlo_ES_Output.csv

Monte Carlo ES Summary:
                Metric  ES_Percentage    ES_Monetary  VaR_Cutoff  Simulations
0  MonteCarlo_ES_97.5%       0.010776 1077647.762878    0.009072        20000


In [ ]:
#Stress ES Calibration Engine (Basel FRTB Mandatory requirement) Module 4.3
#Mandatory additional requiremenrt for Basel FRTB
# market risk capital must reflect crisis like environments not just to recent calm conditions #ES under stresss mkt period

#load Historical portfolio loss series
data = pd.read_csv("Historical_VaR_Output.csv", index_col=0)

losses = data["Portfolio_Loss"].dropna()

print("\nLoss Dataset Loaded Successfully:")
print(losses.describe())
#




Loss Dataset Loaded Successfully:
count   1303.000000
mean      -0.000190
std        0.004605
min       -0.024238
25%       -0.002663
50%       -0.000223
75%        0.001840
max        0.027289
Name: Portfolio_Loss, dtype: float64


In [ ]:
#Basel FRTB  ES Confidence Level
#Stress Expected loss is still computed at Basel ES level

Confidence_level = 0.975

print ("\nBasel FRTB ES Confidence Level:")
print(Confidence_level)




Basel FRTB ES Confidence Level:
0.975


In [ ]:
#Identify Stress Period Observations
# stress windows are chosen based on historical crisis  2008 GFC example
# robust quantitative approach for this example = define stress regime as the worse 10% of portfoio loss dist.
# (represents crisis like tail behavior), extreme stress mkt conditiond for calibration

stress_threshold = np.quantile(losses, 0.90)

stress_losses = losses[losses >= stress_threshold]

print ("\nStress Threshold (Top 10% Loss Days):", stress_threshold)
print("\nNumber of Stress Observations in Period:" , len(stress_losses))







Stress Threshold (Top 10% Loss Days): 0.0051636621039971384

Number of Stress Observations in Period: 131


In [ ]:
#Current expected Shortfall
#ES under full loss distribution

VaR_cutoff_current = np.quantile(losses, Confidence_level)
tail_losses_current = losses[losses >= VaR_cutoff_current]

Current_ES = tail_losses_current.mean()

print("\nCurrent Expected Shortfall (97.5%):" , Current_ES)
#


Current Expected Shortfall (97.5%): 0.013668698694334165


In [ ]:
#Stress Expected Shortfall

VaR_cutoff_stress = np.quantile(stress_losses, Confidence_level)
tail_losses_stress = stress_losses[stress_losses >= VaR_cutoff_stress]

Stress_ES = tail_losses_stress.mean()

print("\nStress Expected Shortfall (97.5%):" , Stress_ES)
#


Stress Expected Shortfall (97.5%): 0.023996084541652352


In [ ]:
#Stress Multiplier Analysis

# tells how much larger tail losses become under stress calibrration
# regulators expect Stress ES to be  sig. higher than currentr ES
#  multiplier is  key driver of trading book capital conservatism under BASEL FRTB

stress_multiplier = Stress_ES / Current_ES

print("\nStress Multiplier (Stress ES / Current ES):" , stress_multiplier)






Stress Multiplier (Stress ES / Current ES): 1.7555500401511528


In [ ]:
# Monetary Stress ES

# how stress calibration is presented in risk commmittees
# basis for conservative capital bufers



portfolio_value = 100_000_000

Current_ES_money = Current_ES * portfolio_value
Stress_ES_money = Stress_ES * portfolio_value

print("\nCurrent Expected Shortfall in USD:" , Current_ES_money)
print("\nStress Expected Shortfall in USD:" , Stress_ES_money)


Current Expected Shortfall in USD: 1366869.8694334165

Stress Expected Shortfall in USD: 2399608.454165235


In [ ]:
#Export Stress ES Calibration Output

#Key Basel FRTB deliverable

stress_es_report = pd.DataFrame({
    "Metric":["Current_ES_97.5%", "Stress_ES_97.5%"],
    "ES_Percentage":[Current_ES, Stress_ES],
    "ES_Monetary":[Current_ES_money, Stress_ES_money],
    "VaR_Cutoff":[VaR_cutoff_current, VaR_cutoff_stress],
    "Stress_Multiplier":[1.0, stress_multiplier]
})

stress_es_report.to_csv("Stress_ES_Calibration_Report.csv" , index=False)

print ("\nStress ES Calibration Report Saved Successfully")
print(" - Stress_ES_Calibration_Report.csv")

print("\nStress ES Calibration Summary:")
print(stress_es_report)


Stress ES Calibration Report Saved Successfully
 - Stress_ES_Calibration_Report.csv

Stress ES Calibration Summary:
             Metric  ES_Percentage    ES_Monetary  VaR_Cutoff  \
0  Current_ES_97.5%       0.013669 1366869.869433    0.009453   
1   Stress_ES_97.5%       0.023996 2399608.454165    0.018992   

   Stress_Multiplier  
0           1.000000  
1           1.755550  


In [ ]:
#Expected Shortfall Dashboard + VaR vs ES Comparison (Basel FRTB Reporting Pack)
#Risk teams build consolidated dashboards , not reported seperately
#Comparing VaR vs ES - providing management ready reporting

print ("ES Dashboard and VaR vs ES Comparison")

#VaR dashbaord from module 3
var_dashboard = pd.read_csv("VaR_Comparison_Dashboard.csv")

#Load Historical ES output module 4.1
es_output = pd.read_csv("Historical_ES_Output.csv")

#Load Monte Carlo ES output module 4.2
montecarlo_es_output = pd.read_csv("MonteCarlo_ES_Output.csv")

#Load Stress ES calibration report module 4.3
stress_es_output = pd.read_csv("Stress_ES_Calibration_Report.csv")

print("\nAll Tail Risk Outputs Loaded Successfully")

#

ES Dashboard and VaR vs ES Comparison

All Tail Risk Outputs Loaded Successfully


In [ ]:
#Consolidated Tail Risk Summary
# combines all the major regulatory risk measures
#presented to trading risk committees, full view of tail exposure under normal and stressed mkt conditions

tail_summary = pd.DataFrame({
    "Risk Measure": [
        "Parametric VaR (99%)",
        "Historical VaR (99%)",
        "Monte Carlo VaR (99%)",
        "Historical ES (97.5%)",
        "Monte Carlo ES (97.5%)",
        "Stress ES (97.5%)",
    ],
    "Value Percentage":[
        var_dashboard.loc[0, "VaR_1day_99%"],
        var_dashboard.loc[1, "VaR_1day_99%"],
        var_dashboard.loc[2, "VaR_1day_99%"],
        es_output.loc[0, "ES_Percentage"],
        montecarlo_es_output.loc[0, "ES_Percentage"],
        stress_es_output.loc[1, "ES_Percentage"]
       ]

    })

print ("\nTail Risk Summary Table:")
print (tail_summary)


Tail Risk Summary Table:
             Risk Measure  Value Percentage
0    Parametric VaR (99%)          0.010713
1    Historical VaR (99%)          0.011574
2   Monte Carlo VaR (99%)          0.010606
3   Historical ES (97.5%)          0.013669
4  Monte Carlo ES (97.5%)          0.010776
5       Stress ES (97.5%)          0.023996


In [ ]:
#Monetary Impact Reporting
# risk measures are translated in money terms, (senior mmanagement reporting)

portfolio_value = 100_000_000 #USD 100M trading portfolio

tail_summary["Value_Monetary_USD"] = tail_summary["Value Percentage"] * portfolio_value

print("\nTail Risk Summary with Monetary Impact:")
print(tail_summary)
#


Tail Risk Summary with Monetary Impact:
             Risk Measure  Value Percentage  Value_Monetary_USD
0    Parametric VaR (99%)          0.010713      1071340.513738
1    Historical VaR (99%)          0.011574      1157447.670971
2   Monte Carlo VaR (99%)          0.010606      1060627.283325
3   Historical ES (97.5%)          0.013669      1366869.869433
4  Monte Carlo ES (97.5%)          0.010776      1077647.762878
5       Stress ES (97.5%)          0.023996      2399608.454165


In [ ]:
#Var vs ES Severity Analysis
# How much additional tail loss exposure exists beyond Var threshold

historical_var = var_dashboard.loc[1, "VaR_1day_99%"]
historical_es_value = es_output.loc[0, "ES_Percentage"]

severity_gap = historical_es_value - historical_var

print("\nVaR vs ES Severity Gap:")
print("Historical VaR 99%:", historical_var)
print("Historical ES 97.5%:", historical_es_value)
print("Tail Severity Gap (ES - VaR):", severity_gap)


VaR vs ES Severity Gap:
Historical VaR 99%: 0.011574476709714
Historical ES 97.5%: 0.0136686986943341
Tail Severity Gap (ES - VaR): 0.002094221984620101


In [ ]:
#Stress ES  Conservatism Analysis
# stress ES calibration is required for regulators
# stress multiplier = how much tail risk increases during stress markets

current_es = stress_es_output.loc[0,"ES_Percentage"]
stress_es_values = stress_es_output.loc[1,"ES_Percentage"]

stress_multiplier = stress_es_values / current_es

print("\nStress ES Conservatism:")
print("Current_ES:", current_es)
print("Stress ES:", stress_es_values)
print("Stress Multiplier:", stress_multiplier)


Stress ES Conservatism:
Current_ES: 0.0136686986943341
Stress ES: 0.0239960845416523
Stress Multiplier: 1.7555500401511572


In [ ]:
#Export Management Dashboard
#full RTFB tail risk reporting infrastructure

tail_summary.to_csv("ES_Reporting_Dashboard.csv", index=False)

print("\nTail Risk Dashboard Saved Successfully")
print(" - ES_Reporting_Dashboard.csv")




Tail Risk Dashboard Saved Successfully
 - ES_Reporting_Dashboard.csv


In [ ]:
# Liquidity Adjusted ES Engine
# baseline ES represents tail risk under standard horizon
# basel requires scaling es depending on how long each risk factor
# takes to liquidate


histrical_es_output = pd.read_csv("Historical_ES_Output.csv")

current_es_value = histrical_es_output.loc[0, "ES_Percentage"]

print("\nCurrent Expected Shortfall (Baseline ES 97.5%):" , current_es_value)


Current Expected Shortfall (Baseline ES 97.5%): 0.0136686986943341


In [ ]:
# BASEL LIQUIDITY HORIZON BUCKETS
# each risk factor assigned horizon in days

liquidity_horizons = {
    "Equity_index":10,
    "Rates_Bond_Index":20,
    "FX":10,
    "Commodity":40
}
print ("\nLiquidity Horizon Mapping (Days):")
for rf, lh in liquidity_horizons.items():
    print(f"{rf}: {lh} days")







Liquidity Horizon Mapping (Days):
Equity_index: 10 days
Rates_Bond_Index: 20 days
FX: 10 days
Commodity: 40 days


In [ ]:
# COMPUTE LIQUIDITY SCALING MULTIPLIERS
# adjustment uses sqrt of time scaling
# how Basel increases the conservatism for illiquid exposures

scaling_factors = {}

for rf, lh in liquidity_horizons.items():
    scaling_factors[rf] = np.sqrt(lh / 10)

print("\nLiquidity Scaling Factors:")
for rf, factor in scaling_factors.items():
    print(rf, "Scaling Factor", factor)



Liquidity Scaling Factors:
Equity_index Scaling Factor 1.0
Rates_Bond_Index Scaling Factor 1.4142135623730951
FX Scaling Factor 1.0
Commodity Scaling Factor 2.0


In [ ]:
# RISK FACTOR ES CONTRIBUTIONS (ILLUSTRATION)
# assigned weights across risk factors, adjusted based on liquidity horizon

risk_factor_weights = pd.read_csv("portfolio_weights.csv", index_col=0)

print("\nRisk Factor Weights:")
print(risk_factor_weights)




Risk Factor Weights:
                       weight
CN_Equity_SSE        0.040000
BRENT_CRUDE_FUT      0.070000
EURUSD               0.080000
GBPUSD               0.060000
GOLD_FUT             0.060000
GSCI_FUT             0.020000
USDCHF              -0.010000
USDJPY              -0.030000
US_BOND_AGG          0.250000
UK_Equity_FTSE100    0.030000
US_Equity_SP500      0.280000
HK_Equity_HSI        0.030000
KR_Equity_KS11       0.030000
JP_Equity_Nikkei225  0.030000
EU_Equity_STOXX50E   0.060000


In [ ]:
# COMPUTE LIQUIDITY-ADJUSTED ES
# Baseline ES scaled for each factor using LH mulitplier for a conservative ES contrubtion for each factor

adjusted_es_contributions = {}

# Map individual risk factors to their liquidity categories
risk_factor_to_liquidity_category = {
    "US_Equity_SP500": "Equity_index",
    "EU_Equity_STOXX50E": "Equity_index",
    "CN_Equity_SSE": "Equity_index",
    "JP_Equity_Nikkei225": "Equity_index",
    "KR_Equity_KS11": "Equity_index",
    "HK_Equity_HSI": "Equity_index",
    "UK_Equity_FTSE100": "Equity_index",
    "US_BOND_AGG": "Rates_Bond_Index",
    "EURUSD": "FX",
    "GBPUSD": "FX",
    "USDJPY": "FX",
    "USDCHF": "FX",
    "GOLD_FUT": "Commodity",
    "GSCI_FUT": "Commodity",
    "BRENT_CRUDE_FUT": "Commodity"
}

# Calculate the sum of weights for each liquidity category
category_sum_of_weights = {category: 0.0 for category in liquidity_horizons.keys()}
for rf_name, category in risk_factor_to_liquidity_category.items():
    if rf_name in risk_factor_weights.index: # Ensure the risk factor exists in the weights
        category_sum_of_weights[category] += risk_factor_weights.loc[rf_name, 'weight']

for rf_category_name in liquidity_horizons.keys():
  base_contribution = current_es_value * category_sum_of_weights[rf_category_name]
  adjusted_contribution = base_contribution * scaling_factors[rf_category_name]
  adjusted_es_contributions[rf_category_name] = adjusted_contribution

print("\nLiquidity Adjusted ES Contributions:")
for rf, value in adjusted_es_contributions.items():
  print(rf,"Adjusted ES:", value)



Liquidity Adjusted ES Contributions:
Equity_index Adjusted ES: 0.006834349347167052
Rates_Bond_Index Adjusted ES: 0.004832614768379676
FX Adjusted ES: 0.0013668698694334104
Commodity Adjusted ES: 0.00410060960830023


In [ ]:
# PORTFOLIO LIQUIDITY-ADJUSTED ES
# aggregated across all adjusted contributions
# Basel requires Capital conservatism for liquidation risk

portfolio_LH_ES = sum(adjusted_es_contributions.values())

print("\nPortfolio Liquidity Adjusted ES (Total):", portfolio_LH_ES)


Portfolio Liquidity Adjusted ES (Total): 0.01713444359328037


In [ ]:
# EXPORT LIQUIDITY HORIZON ES REPORT
# Deliverable report required  under Basel Internal model approach

lh_es_report = pd.DataFrame({
    "Risk Factor": list(liquidity_horizons.keys()),
    "Liquidity_Horizon_Days": list(liquidity_horizons.values()),
    "Scaling Factor": list(scaling_factors.values()),
    "Adjusted_ES_Contribution":(adjusted_es_contributions.values())
})

lh_es_report.loc["TOTAL"] = ["Portfolio_Total", "-", "_" , portfolio_LH_ES]

lh_es_report.to_csv("Liquidity_Adjusted_ES_Output.csv", index=False)

print("\nExport Saved Successfully")
print(" - Liquidity_Horizon_ES_Report.csv")

print("\nLiquidity Adjusted ES Report:")
print(lh_es_report)


Export Saved Successfully
 - Liquidity_Horizon_ES_Report.csv

Liquidity Adjusted ES Report:
            Risk Factor Liquidity_Horizon_Days Scaling Factor  \
0          Equity_index                     10       1.000000   
1      Rates_Bond_Index                     20       1.414214   
2                    FX                     10       1.000000   
3             Commodity                     40       2.000000   
TOTAL   Portfolio_Total                      -              _   

       Adjusted_ES_Contribution  
0                      0.006834  
1                      0.004833  
2                      0.001367  
3                      0.004101  
TOTAL                  0.017134  


In [ ]:
#RISK FACTOR ELIGIBILITY TEST (RFE). Module 6.1
# only sufficient observable market data Risk factors are eligible
# Risk factors with limited data require additional conservative capital charges forr intrrnal model approach approach

#load risk factor dataset

returns = pd.read_csv("Market_RiskFactor_Returns.csv", index_col=0)

print("\nRisk Factor Returns Loaded Successfully:")
print(returns.head())



Risk Factor Returns Loaded Successfully:
            CN_Equity_SSE  BRENT_CRUDE_FUT    EURUSD    GBPUSD  GOLD_FUT  \
date                                                                       
2022-01-05      -0.010280         0.009950 -0.001649  0.003566  0.005826   
2022-01-06      -0.002534         0.014620  0.002644  0.001788 -0.019872   
2022-01-07      -0.001825        -0.002931 -0.001470 -0.001625  0.004630   
2022-01-10       0.003898        -0.010823  0.005040  0.004327  0.000779   
2022-01-11      -0.007284         0.034635 -0.002076 -0.000883  0.011170   

            GSCI_FUT    USDCHF    USDJPY  US_BOND_AGG  UK_Equity_FTSE100  \
date                                                                       
2022-01-05  0.004097 -0.002311  0.007309     0.000000           0.001558   
2022-01-06  0.008057  0.001287 -0.000396     0.000000          -0.008886   
2022-01-07  0.000345  0.004362 -0.002267     0.000000           0.004673   
2022-01-10 -0.007098 -0.002782 -0.002134     

In [ ]:
# BASEL  RFE MODELLABILITY CRITERION (SIMPLIFIED)
# Basel minimum req. aprox : modellable if at lest 200 nin missing daily observations
# observation frequncy determines the criterion ( non sparce or illiquid factors )

min_observations = 200

print ("\nBasel RFE Minimum Observation Threshold:" , min_observations)



Basel RFE Minimum Observation Threshold: 200


In [ ]:
# AVAILABLE OBSERVATIONS PER RISK FACTOR
# less liquid or sparce factors may have missing observations

observation_count = returns.count()

print("\nAvailable Observations per Risk Factor:")
print(observation_count)








Available Observations per Risk Factor:
CN_Equity_SSE          1303
BRENT_CRUDE_FUT        1303
EURUSD                 1303
GBPUSD                 1303
GOLD_FUT               1303
GSCI_FUT               1303
USDCHF                 1303
USDJPY                 1303
US_BOND_AGG            1303
UK_Equity_FTSE100      1303
US_Equity_SP500        1303
HK_Equity_HSI          1303
KR_Equity_KS11         1303
JP_Equity_Nikkei225    1303
EU_Equity_STOXX50E     1303
dtype: int64


In [ ]:
# MODELLABILITY CLASSIFICATION
# if risk factor meets minimum observation count it is modelable, if falls below threshold = non modelable
# if non modelable cannot be included directly in the internal model ES FRTB model approach

modellability = observation_count.apply(
    lambda x: "Modelable" if x >= min_observations else "Non-Modelable"
)


print("\nRisk Factors Modellability Results:")
print(modellability)




Risk Factors Modellability Results:
CN_Equity_SSE          Modelable
BRENT_CRUDE_FUT        Modelable
EURUSD                 Modelable
GBPUSD                 Modelable
GOLD_FUT               Modelable
GSCI_FUT               Modelable
USDCHF                 Modelable
USDJPY                 Modelable
US_BOND_AGG            Modelable
UK_Equity_FTSE100      Modelable
US_Equity_SP500        Modelable
HK_Equity_HSI          Modelable
KR_Equity_KS11         Modelable
JP_Equity_Nikkei225    Modelable
EU_Equity_STOXX50E     Modelable
dtype: object


In [ ]:
# BUILD THE RFE MODELLABILITY REPORT

# Risk teams produce for Basel compleiance
# risk factors, number of avail market observations, modelable vs non modelable classification
# input CapItal Computation and Desk  Model  Approval

rfe_report = pd.DataFrame({
    "Risk Factor": observation_count.index,
    "Observations": observation_count.values,
    "Modellability": modellability.values
})


print("\nRFE Report Preview:")
print(rfe_report)


RFE Report Preview:
            Risk Factor  Observations Modellability
0         CN_Equity_SSE          1303     Modelable
1       BRENT_CRUDE_FUT          1303     Modelable
2                EURUSD          1303     Modelable
3                GBPUSD          1303     Modelable
4              GOLD_FUT          1303     Modelable
5              GSCI_FUT          1303     Modelable
6                USDCHF          1303     Modelable
7                USDJPY          1303     Modelable
8           US_BOND_AGG          1303     Modelable
9     UK_Equity_FTSE100          1303     Modelable
10      US_Equity_SP500          1303     Modelable
11        HK_Equity_HSI          1303     Modelable
12       KR_Equity_KS11          1303     Modelable
13  JP_Equity_Nikkei225          1303     Modelable
14   EU_Equity_STOXX50E          1303     Modelable


In [ ]:
# EXPORT RFE REPORT
# Basel FRTB deliverable


rfe_report.to_csv("RFE_Modellability_Report.csv", index=False)

print("\nRFE Report Saved Successfully")
print(" - RFE_Modellability_Report.csv")


RFE Report Saved Successfully
 - RFE_Modellability_Report.csv


In [ ]:
# NON MODELABLE  RISK FACTORS (NMRF), CAPITAL ADD-ON ENGINE

# NMFR CaPital Add-on Framework
# Some exposures in stress markets are illiquid, exotic derivatives etc., statistical models become unrealibale (sparse data )
# for Internal models ES framework Basel requires additional conservative capital add-on  ( NMRF Capital Add-on)


rfe_report= pd.read_csv("RFE_Modellability_Report.csv")

print("RFE Report Loaded Successfully\n")
print(rfe_report)





RFE Report Loaded Successfully

            Risk Factor  Observations Modellability
0         CN_Equity_SSE          1303     Modelable
1       BRENT_CRUDE_FUT          1303     Modelable
2                EURUSD          1303     Modelable
3                GBPUSD          1303     Modelable
4              GOLD_FUT          1303     Modelable
5              GSCI_FUT          1303     Modelable
6                USDCHF          1303     Modelable
7                USDJPY          1303     Modelable
8           US_BOND_AGG          1303     Modelable
9     UK_Equity_FTSE100          1303     Modelable
10      US_Equity_SP500          1303     Modelable
11        HK_Equity_HSI          1303     Modelable
12       KR_Equity_KS11          1303     Modelable
13  JP_Equity_Nikkei225          1303     Modelable
14   EU_Equity_STOXX50E          1303     Modelable


In [ ]:
# EXTRACT NON MODELABLE RISK FACTORS (NMRF)
# Factors generate capital charges, impact trading reg. capital

nmrf_factors = rfe_report[
    rfe_report["Modellability"] == "Non-Modelable"
]["Risk Factor"].tolist()

print("\nNon-Modelable Risk Factors Identified:")
print(nmrf_factors)
print("Count", len(nmrf_factors))


Non-Modelable Risk Factors Identified:
[]
Count 0


In [ ]:
# BASEL CONSERVATIVE STRESS SHOCK MULTIPLIER
# NMRF not treated with same ES model, Basel requires conservative stress based add-on
# involes stress calibration and liquidity addressabels
# priciple: non modelable exposures requires extreme conservative shocks
# here we implement a clean approximation
# a stress multiplier with 3 standard deviations

stress_multiplier = 3.0

print("\nBasel Conservative Stress Shock Multiplier Applied:", stress_multiplier)




Basel Conservative Stress Shock Multiplier Applied: 3.0


In [ ]:
# COMPUTE FACTOR-LEVEL NMRF CAPITAL ADD-ON

# Volatility estimation for the NRMF
# need a measure of risk magnitude
# daily vol of factor X stress shock factor to procuce add-on capital charge

nmrf_results = []

for factor in nmrf_factors:

  # estimate vol from avail. returns
    factor_vol = returns[factor].std()

  # conservative Basel stress charge aproximation
    nmrf_charge =  stress_multiplier * factor_vol

    nmrf_results.append({
        "Risk_Factor": factor,
        "Daily_Volatility": factor_vol,
        "Stress_Multiplier": stress_multiplier,
        "NMRF_Capital_Add-on": nmrf_charge

    })


# convert results into DataFrame
nmrf_report = pd.DataFrame(nmrf_results)

print ("\nNMRF Capital Add-on Report:")
print(nmrf_report)







NMRF Capital Add-on Report:
Empty DataFrame
Columns: []
Index: []


In [ ]:
# TOTAL NMRF  CAPITAL ADD-ON (ROBUST BASEL VERSION)
# NMRF capital add-on per factor & entire portfilo
# aggregation also required, all NMRF factors for entire portfilo
# Basel Trading Book Capital is driven by : ES, Stress ES, Liquidity horizon adjustemtns, and nmrf add-on



if nmrf_report.empty:

    print("\nNo NMRF Detected in this Porfolio.")
    print("Therefore, Basel NMRF Capital Add-on is Zero.\n")

    total_nmrf_charge= 0.0

    # create placeholder report for documentation consistency
    nmrf_report = pd.DataFrame({
      "Risk_Factor": ["None"],
      "Daily_Volatility": [0.0],
      "Stress_Multiplier": [stress_multiplier],
      "NMRF_Capital_Add-on": [0.0]
    })


else:
  total_nmrf_charge = nmrf_report["NMRF_Capital_Add-on"].sum()


print("\nTotal Portfolio NMRF Capital Add-on (Robust Basel Version):", total_nmrf_charge)


No NMRF Detected in this Porfolio.
Therefore, Basel NMRF Capital Add-on is Zero.


Total Portfolio NMRF Capital Add-on (Robust Basel Version): 0.0


In [ ]:
# EXPORT BASEL NMRF ADD-ON REPORT
# Audit ready Basel deliverable
# Part of full Internal Model Approach Capital workflow

nmrf_report.to_csv("NMRF_Capital_Add-on.csv", index=False)

print("\nNMRF Capital Add-on Report Saved Successfully")
print(" - NMRF_Capital_Add-on_Report.csv")

print("\nFinal NMRF Report:")
print(nmrf_report)


NMRF Capital Add-on Report Saved Successfully
 - NMRF_Capital_Add-on_Report.csv

Final NMRF Report:
  Risk_Factor  Daily_Volatility  Stress_Multiplier  NMRF_Capital_Add-on
0        None          0.000000           3.000000             0.000000


In [ ]:
# VAR BACKTESTING FRAMEWORK + BASEL EXCEPTION MONITORING
# MODEL VALIDATION
# Model Validation compares Model predicted VaR vs actual realized portfolio losses
# If actual losses exceed VaR model too frequently Model fails and is not accepted

# Under Basel Internal models are allowed only if:
# 1.VaR and ES forecasts are statistically reliable
# 2.Actual trading losses do not exceed VaR too frequently
# 3.Exceptions are monitored and reported
# BACKTESTING is key model governace requirement under Basel and
# alligns with SR 11.7 Model Risk Management expectations

# VaR Backtesting Framework. Module 7

# Load Historical Portfolio Losses (Actual Loss Series)
loss_data = pd.read_csv("Historical_VaR_Output.csv", index_col=0)

# Load VaR Forecast Value (Parametric or Historical)
var_dashboard = pd.read_csv("VaR_Comparison_Dashboard.csv", index_col=0)

# Select Historical VaR(99%)
VaR_99 = var_dashboard.loc["Historical VaR", "VaR_1day_99%"]

print("\nHistorical VaR Forecast(99%):" , VaR_99)



Historical VaR Forecast(99%): 0.011574476709714


In [ ]:
# ACTUAL LOSS SERIES
# Represents trading desk historical daily P/L reporting systems

actual_losses = loss_data["Portfolio_Loss"].dropna()

print("\nActual Loss Series Loaded:")
print(actual_losses.head())



Actual Loss Series Loaded:
date
2022-01-05    0.005320
2022-01-06    0.002257
2022-01-07    0.000561
2022-01-10    0.001404
2022-01-11   -0.006105
Name: Portfolio_Loss, dtype: float64


In [ ]:
# IDENTIFY VAR EXCEPTIONS (BREACHES)
# A Breach occurs when:
# Loss>VaR
# Basel expects approx 1% exception frequerncy for a VaR 99% Model.



exceptions = actual_losses[actual_losses > VaR_99]

num_exceptions = len(exceptions)
total_days = len(actual_losses)

print ("\nTotal Backtesting Days", total_days)
print("Total VaR Exceptions Detected:", num_exceptions)




Total Backtesting Days 1303
Total VaR Exceptions Detected: 20


In [ ]:
# EXCEPTION RATE
# Basel uses this rate to classify VaR Models into green, yellow or red zones ( basis of Basel traffic light framework )

expection_rate = num_exceptions / total_days

print("\nException Rate:", expection_rate)



Exception Rate: 0.015349194167306216


In [ ]:
 # BUILD BACKTESTING REPORT TABLE

 # Produced for Regulatory Backtesting,
 # Documents realized losses, VaR forecast and Breaches,
 # Part of Model Risk Goverance documentation and SR11-7

backtest_report = pd.DataFrame({
    "Date_Index": actual_losses.index,
    "Actual_Loss": actual_losses.values,
    "VaR_Forecast_99": VaR_99,
    "Exception_flag": actual_losses > VaR_99
 })

print("\nBacktesting Report preview:")
print(backtest_report.head(100))






Backtesting Report preview:
            Date_Index  Actual_Loss  VaR_Forecast_99  Exception_flag
date                                                                
2022-01-05  2022-01-05     0.005320         0.011574           False
2022-01-06  2022-01-06     0.002257         0.011574           False
2022-01-07  2022-01-07     0.000561         0.011574           False
2022-01-10  2022-01-10     0.001404         0.011574           False
2022-01-11  2022-01-11    -0.006105         0.011574           False
...                ...          ...              ...             ...
2022-05-09  2022-05-09     0.017685         0.011574            True
2022-05-10  2022-05-10     0.003124         0.011574           False
2022-05-11  2022-05-11    -0.004356         0.011574           False
2022-05-12  2022-05-12     0.003104         0.011574           False
2022-05-13  2022-05-13    -0.013892         0.011574           False

[100 rows x 4 columns]


In [ ]:
# EXPORT VAR BACKTESTING REPORT
# Basel validation deliverable
# Reviewed by Model Validation Team, Internal Audit and Regulators

backtest_report.to_csv("VaR_Backtesting_Report.csv", index=False)

print("\nBacktesting Report Saved Successfully")
print(" - VaR_Backtesting_Report.csv")


Backtesting Report Saved Successfully
 - VaR_Backtesting_Report.csv


In [ ]:
# BASEL TRAFFIC LIGHT FRAMEWORK FOR VAR BACKTESTING (CAPITAL MULTIPLIERS)
# Load file and confirm structure of dataset


#Load VaR Backtesting Report Dataset
bt = pd.read_csv("VaR_Backtesting_Report.csv", index_col=0)

print("\nBacktesting Report Loaded Successfully:")
print("Columns Available:", bt.columns)

print("\nDataset preview:")
print(bt.head(100))






Backtesting Report Loaded Successfully:
Columns Available: Index(['Actual_Loss', 'VaR_Forecast_99', 'Exception_flag'], dtype='object')

Dataset preview:
            Actual_Loss  VaR_Forecast_99  Exception_flag
Date_Index                                              
2022-01-05     0.005320         0.011574           False
2022-01-06     0.002257         0.011574           False
2022-01-07     0.000561         0.011574           False
2022-01-10     0.001404         0.011574           False
2022-01-11    -0.006105         0.011574           False
...                 ...              ...             ...
2022-05-09     0.017685         0.011574            True
2022-05-10     0.003124         0.011574           False
2022-05-11    -0.004356         0.011574           False
2022-05-12     0.003104         0.011574           False
2022-05-13    -0.013892         0.011574           False

[100 rows x 3 columns]


In [ ]:
# COUNT TOTAL VAR EXCEPTIONS
# Basel evaluates Exception over a rolling 1 year window (250 days), key reg. imput into Basel traffic light clasisfication

exceptions = bt["Exception_flag"].sum()
total_days = len(bt)

print("\nTotal Backtesting Days:", total_days)
print("Total VaR Exceptions Detected:", exceptions)


Total Backtesting Days: 1303
Total VaR Exceptions Detected: 20


In [ ]:
# COUNT TOTAL VAR EXCEPTIONS (ALTERNATIVE COUNT, WINDOW COUNT =250)
# Exceptions for the last 250 days ( last year of dataset)

# Define the window size
window_size = 250

# Sum only the last 250 entries for 'exceptions'
exceptions_last_250_days = bt["Exception_flag"].tail(window_size).sum()
total_days_for_calculation = window_size

print(f"\nTotal Backtesting Days (last {window_size} entries):", total_days_for_calculation)
print(f"Total VaR Exceptions Detected (last {window_size} entries):", exceptions_last_250_days)


Total Backtesting Days (last 250 entries): 250
Total VaR Exceptions Detected (last 250 entries): 3


In [ ]:
# BASEL TRAFFIC LIGHT ZONE CLASSIFICATION
# GREEN ZONE= 0-4 Exceptions, YELLOW ZONE= 5-9 Exceptions, RED ZONE= 10 or more Exceptions

if exceptions_last_250_days <= 4:
  zone = "GREEN"
  interpretation = "Model Performance Acceptable under Basel"
  multiplier = 1.0

elif exceptions_last_250_days <= 9.0:
  zone = "YELLOW"
  interpretation = "Model Under Review, Capital Multiplie Increases"
  multiplier = 1.5

else:
  zone = "RED"
  interpretation = "Model Unacceptable, Internal Model approval at risk"
  multiplier = 2.0

print("\nBasel Trafffic Light Zone Classification:")
print("Zone:", zone)
print("Interpretation:", interpretation)
print("Capital Multiplier Applied:", multiplier)


Basel Trafffic Light Zone Classification:
Zone: GREEN
Interpretation: Model Performance Acceptable under Basel
Capital Multiplier Applied: 1.0


In [ ]:
# consider a rolling 1-year window (typically 250 trading days)
# For Basel traffic light classification
# The previous calculation was based on the entire historical period

# Load VaR Backtesting Report Dataset
bt = pd.read_csv("VaR_Backtesting_Report.csv", index_col=0, parse_dates=True)

# Define the rolling window size for Basel (typically 250 trading days for a year)
window_size = 250

# List to store results for each rolling window
rolling_backtest_results = []

# Iterate through the DataFrame using a rolling window
for i in range(window_size - 1, len(bt)):
    current_window = bt.iloc[i - window_size + 1 : i + 1]

    window_start_date = current_window.index[0]
    window_end_date = current_window.index[-1]

    # Count exceptions in the current window using the sum of the Exception_flag
    exceptions_in_window = current_window["Exception_flag"].sum()
    total_days_in_window = len(current_window)

    zone = ""
    interpretation = ""
    multiplier = 0.0

    # Apply Basel Traffic Light Zone Classification
    if exceptions_in_window <= 4:
        zone = "GREEN"
        interpretation = "Model Performance Acceptable under Basel"
        multiplier = 1.0
    elif exceptions_in_window <= 9: # Basel standard yellow zone is 5-9 exceptions
        zone = "YELLOW"
        interpretation = "Model Under Review, Capital Multiplier Increases"
        multiplier = 1.5
    else: # 10 or more exceptions
        zone = "RED"
        interpretation = "Model Unacceptable, Internal Model approval at risk"
        multiplier = 2.0

    rolling_backtest_results.append({
        "Window_End_Date": window_end_date,
        "Window_Start_Date": window_start_date,
        "Total_Days_in_Window": total_days_in_window,
        "Exceptions_in_Window": exceptions_in_window,
        "Zone": zone,
        "Interpretation": interpretation,
        "Capital_Multiplier": multiplier
    })

# Convert results to a DataFrame for better display
rolling_report_df = pd.DataFrame(rolling_backtest_results)

print(f"\nBasel Traffic Light Zone Classification over Rolling {window_size}-day Windows:")
# Display the first few and last few results to show the rolling nature
print(rolling_report_df.head())
print("\n...")
print(rolling_report_df.tail())

# Optionally, save the rolling report
rolling_report_df.to_csv("VaR_Backtesting_Rolling_Report.csv", index=False)
print("\nVaR Backtesting Rolling Report Saved Successfully to VaR_Backtesting_Rolling_Report.csv")

# Also, provide the classification for the very last window explicitly as it represents the most current assessment
last_window_result = rolling_backtest_results[-1]
print("\n--- Latest Rolling Window Classification ---")
print(f"Period: {last_window_result['Window_Start_Date'].strftime('%Y-%m-%d')} to {last_window_result['Window_End_Date'].strftime('%Y-%m-%d')}")
print(f"Total Days: {last_window_result['Total_Days_in_Window']}")
print(f"Exceptions: {last_window_result['Exceptions_in_Window']}")
print(f"Zone: {last_window_result['Zone']}")
print(f"Interpretation: {last_window_result['Interpretation']}")
print(f"Capital Multiplier: {last_window_result['Capital_Multiplier']}")


Basel Traffic Light Zone Classification over Rolling 250-day Windows:
  Window_End_Date Window_Start_Date  Total_Days_in_Window  \
0      2022-11-07        2022-01-05                   250   
1      2022-11-08        2022-01-06                   250   
2      2022-11-09        2022-01-07                   250   
3      2022-11-10        2022-01-10                   250   
4      2022-11-11        2022-01-11                   250   

   Exceptions_in_Window Zone  \
0                    10  RED   
1                    10  RED   
2                    10  RED   
3                    10  RED   
4                    10  RED   

                                      Interpretation  Capital_Multiplier  
0  Model Unacceptable, Internal Model approval at...            2.000000  
1  Model Unacceptable, Internal Model approval at...            2.000000  
2  Model Unacceptable, Internal Model approval at...            2.000000  
3  Model Unacceptable, Internal Model approval at...            2.000

In [ ]:
# BUILD BASEL TRAFFIC LIGHT REPORT

# Formal Basel summary report produced by the Model Validation Team
# Essential For Desk level Model governance and Audit documentation

traffic_light_report = pd.DataFrame({
    "Backtesting_Window_Days": [total_days_for_calculation],
    "Total_Exceptions": [exceptions_last_250_days],
    "Traffic_Light_zone": [zone],
    "Regulatory_Interpretation": [interpretation],
    "Capital_Multiplier": [multiplier]
})

print("\nBasel Traffic Light Summary Report:")
print(traffic_light_report)
#


Basel Traffic Light Summary Report:
   Backtesting_Window_Days  Total_Exceptions Traffic_Light_zone  \
0                      250                 3              GREEN   

                  Regulatory_Interpretation  Capital_Multiplier  
0  Model Performance Acceptable under Basel            1.000000  


In [ ]:
# EXPORT REPORT DELIVERABLE
# Audit ready regulatory deliverable
# Documents how trading desk model performs under Basel backtesting rules
# If remains elligible for Interal Model usage

traffic_light_report.to_csv("Basel_Traffic_Light_Report.csv", index=False)

print("\nBasel Traffic Light Report Saved Successfully")
print(" - Basel_Traffic_Light_Report.csv")


Basel Traffic Light Report Saved Successfully
 - Basel_Traffic_Light_Report.csv


In [ ]:
# BASEL FRTB INTERNAL MODELS CAPITAL CHARGE WORKFLOW (END TO END CAPITAL COMPUTATION)
# BASEL FRTB CAPITAL WORKFLOW (Module 8)
# Basel componets into one final Trading Book Capital requirement
# Management ready capital charge Report ( used by Market Risk and Capital Planning teams)
# From individual risk measures to final Regulatory Capital Requirements

# Load ES outputs
current_es = pd.read_csv("Historical_ES_Output.csv", index_col=0)
stress_es = pd.read_csv("Stress_ES_Calibration_Report.csv", index_col=0)

# Load Liquidity Horizon Adjusted ES
lh_es = pd.read_csv("Liquidity_Adjusted_ES_Output.csv", index_col=0)

# Load NMRF Add-On Output
nmrf = pd.read_csv("NMRF_Capital_Add-on.csv", index_col=0)

# Load Basel Traffic Light Multiplier
traffic_light = pd.read_csv("Basel_Traffic_Light_Report.csv", index_col=0)

print("\nAll Basel Capital Components Loaded Successfully:")

# Load Historical Portfolio Losses (Actual Loss Series)
loss_data = pd.read_csv("Historical_VaR_Output.csv", index_col=0)









All Basel Capital Components Loaded Successfully:


In [ ]:
# EXTRACT CAPITAL INPUTS
# Capturing all quantitative intputs into final Capital Charge
# performed by trading book capital team

# Tail Risk under today's market conditions
current_es_value = current_es.iloc[0]["ES_Percentage"]

# Tail Risk under crisis calibration
stress_es_value = stress_es.iloc[1]["ES_Percentage"]

# liquidiy horizon Adjusted Expected shortfall
# (Increases capital for exposure that cannot be liquidated quickly)
lh_total = lh_es.iloc[-1]["Adjusted_ES_Contribution"]

# captures extra buffers fon non modelable risk factors
nmrf_total = nmrf["NMRF_Capital_Add-on"].sum()

# Basel gtraffic light mmultiplier which reflects backtesting performance
multiplier = traffic_light.iloc[0]["Capital_Multiplier"]

print("\nBasel Capital Inputs Extracted Successfully:")

print("Current ES", current_es_value)
print("Stress ES", stress_es_value)
print("Liquidity Horizon Adjustemnt", lh_total)
print("Total NMRF Add-On", nmrf_total)
print("Backtesting Multiplier", multiplier)







Basel Capital Inputs Extracted Successfully:
Current ES 0.0136686986943341
Stress ES 0.0239960845416523
Liquidity Horizon Adjustemnt 0.0171344435932803
Total NMRF Add-On 0.0
Backtesting Multiplier 1.0


In [ ]:
# BASEL ES CAPITAL BASE
# key conservative Basel FTRB rule
# BANKS MUST NOT BASE CAPITAL ONLY ON CURRENT MARKET CONDITIONS
# BASEL INFORMS THAT CAPITAL MUST REMAIN ANCHORED TO STRESS CONDITIONS ALSO
# ES CAPITAL BASE IS DEFINED AS THE MAXIMUM OF CURRENT ES & STRESS ES
# ENSURES CAPITAL REFLECTS THE CRISIS LIKE TAIL RISK

base_es_capital = max(current_es_value, stress_es_value)

print("\nBasel ES Capital Base(Max of Current and Stress):", base_es_capital)










Basel ES Capital Base(Max of Current and Stress): 0.0239960845416523


In [ ]:
# LIQUIDITY HORIZON OVERLAY
# Basel assigns LH buckets and scales ES accordingly
# Adjustemtn increasing capital for illiquid exposures
# Making the framework more realistic during stress periods
# After scaling Capital Requirement becomes more conservative

lh_adjusted_capital = base_es_capital + lh_total

print("\nCapital asfter Liquidity Horizon Adjustment:", lh_adjusted_capital)



Capital asfter Liquidity Horizon Adjustment: 0.0411305281349326


In [ ]:
# ADD NMRF CAPITAL CHARGE
# exposures with insuffient market data to model tail risk reliably
# Basel requires additional conservative buffer to such risk factors (data sparse exposures)
# Largest driver of capital inpact for complex trading desks


capital_with_nmrf = lh_adjusted_capital + nmrf_total

print("\nCapital After Adding NMRF ADD-On", capital_with_nmrf)




Capital After Adding NMRF ADD-On 0.0411305281349326


In [ ]:
# APPLY TRAFFIC LIGHT MULTIPLIER
# Where Model Validation directly impacts Capital (GREEN ZONE multiplier = 1)
# Weak Model performace translates to higher reg. capital
# Key Supervisory discipline mechanism

final_capital_charge = capital_with_nmrf * multiplier

print("\nFinal Basel FRTB Internal Model Capital Charge:", final_capital_charge)


Final Basel FRTB Internal Model Capital Charge: 0.0411305281349326


In [ ]:
# EXPORT MANAGEMENT-READY CAPITAL REPORT

# Capital reporting produced by market risk and planning teams
# For regulatory submission, Icap process, Senior Risk Committees and Trading Desk cap. allocation

capital_report = pd.DataFrame({
    "Component": [
        "Current Expected Shortfall",
        "Stress Expected Shortfall",
        "Basel ES Capital Base",
        "Liquidity Horizon Adjustment",
        "NMRF Add-on Capital",
        "Backtesting Multiplier",
        "Final Capital Charge"

],
    "Value": [
        current_es_value,
        stress_es_value,
        base_es_capital,
        lh_total,
        nmrf_total,
        multiplier,
        final_capital_charge
    ]
})

capital_report.to_csv("FRTB_InternalModel_Capital_Report.csv", index=False)

print("\nBasel Capital Charge Report Saved Successfully")
print(" - Basel_Capital_Charge_Report.csv")




Basel Capital Charge Report Saved Successfully
 - Basel_Capital_Charge_Report.csv
